<a href="https://colab.research.google.com/github/fares3010/classification/blob/main/classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Overview: Advanced Behavioral Analysis using GRU Networks

This project implements a comprehensive pipeline for analyzing complex animal behavior from video tracking data, focusing on memory-optimized processing, robust feature engineering, and advanced machine learning for classification. The core objective is to detect and classify specific behavioral patterns, including rare and challenging events like aggression, by leveraging Recurrent Neural Networks (GRU) and sophisticated data handling techniques.

**Key Stages of the Pipeline:**

1.  **Data Ingestion and Preprocessing:**
    *   Loads video metadata, behavioral annotations, and raw tracking data (e.g., body part coordinates).
    *   Handles missing data through interpolation and filtering.
    *   Optimizes DataFrame memory usage to efficiently manage large datasets.
    *   Downsamples tracking data to a consistent frame rate.

2.  **Feature Engineering:**
    *   Transforms raw body part coordinates into meaningful kinematic and spatial features (e.g., distances between body parts, angles, speeds, accelerations, angular velocities, distances to arena walls and center).
    *   Calculates derived metrics to capture subtle behavioral cues.

3.  **Behavioral Annotation and Windowing:**
    *   Processes human-labeled annotations to identify distinct behavioral segments.
    *   Inserts 'other_behavior' annotations to represent periods of non-specific activity.
    *   Organizes data into fixed-size

This cell imports all the necessary libraries for data processing, model training, and memory optimization. It includes libraries like `pandas` for data manipulation, `numpy` for numerical operations, `torch` for building the GRU model, `sklearn` for data preprocessing and evaluation, and various other utilities for file handling, parallel processing, and memory management.

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Callable, Optional, Any, Union, Generator
from functools import reduce, partial, wraps
from pandas.api.types import is_datetime64_any_dtype, is_numeric_dtype, is_categorical_dtype
import inspect
from openai import OpenAI
import json
import re
import gc
import os
import tempfile
import warnings
from collections import deque
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


This cell defines memory optimization utility functions. It includes:
- `get_memory_usage_mb()`: A function to get the current memory usage of the Python process in MB.
- `optimize_dataframe_memory()`: A function to reduce the memory footprint of a Pandas DataFrame by downcasting numeric types and converting suitable columns to categorical types.
- `clear_memory()`: A function to explicitly force garbage collection to free up memory.
- `chunk_list()`: A utility to split a list into smaller, manageable chunks.

In [ ]:
# =============================================================================
# MEMORY OPTIMIZATION UTILITIES
# =============================================================================

def get_memory_usage_mb() -> float:
    """Get current memory usage in MB."""
    try:
        import psutil
        process = psutil.Process(os.getpid())
        return process.memory_info().rss / 1024 / 1024
    except ImportError:
        return 0.0


def optimize_dataframe_memory(df: pd.DataFrame, verbose: bool = False) -> pd.DataFrame:
    """
    Optimize DataFrame memory usage by downcasting numeric types and using categories.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to optimize
    verbose : bool
        Print memory reduction stats

    Returns
    -------
    pd.DataFrame
        Memory-optimized DataFrame
    """
    if df.empty:
        return df

    start_mem = df.memory_usage(deep=True).sum() / 1024 / 1024

    for col in df.columns:
        col_type = df[col].dtype

        # Skip categorical columns - they're already optimized
        if isinstance(col_type, pd.CategoricalDtype):
            continue

        # Process numeric columns (int, float)
        if pd.api.types.is_integer_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)

        elif pd.api.types.is_float_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()

            if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)

        elif col_type == object:
            # Object columns - convert to category if low cardinality
            num_unique = df[col].nunique()
            num_total = len(df[col])
            if num_unique / num_total < 0.5:  # Less than 50% unique values
                df[col] = df[col].astype('category')

    end_mem = df.memory_usage(deep=True).sum() / 1024 / 1024

    if verbose:
        reduction = 100 * (start_mem - end_mem) / start_mem if start_mem > 0 else 0
        print(f"Memory: {start_mem:.2f} MB -> {end_mem:.2f} MB ({reduction:.1f}% reduction)")

    return df


def clear_memory(verbose: bool = False) -> None:
    """Force garbage collection and clear memory."""
    if verbose:
        before = get_memory_usage_mb()

    gc.collect()

    if verbose:
        after = get_memory_usage_mb()
        print(f"Memory cleared: {before:.1f} MB -> {after:.1f} MB")


def chunk_list(lst: list, chunk_size: int) -> Generator[list, None, None]:
    """Split a list into chunks of specified size."""
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

This cell contains functions for batch processing large-scale video data, particularly designed for memory efficiency:
- `get_temp_dir()`: Retrieves or creates a temporary directory for storing intermediate results.
- `save_batch_to_disk()`: Saves a processed DataFrame batch to disk as a Parquet file to conserve memory.
- `load_batch_from_disk()`: Loads a batch from a Parquet file on disk.
- `cleanup_temp_files()`: Removes all temporary batch files from the system.
- `process_single_video_safe()`: Processes a single video with robust error handling and memory optimization.
- `process_video_batch()`: Processes a batch of video configurations, saves results to disk, and includes memory management logic.
- `aggregate_batches_from_disk()`: Aggregates all saved batch files from disk into a single master DataFrame, ensuring globally unique window IDs.
- `main_optimized()`: The main memory-optimized function for processing a large number of videos, utilizing batch processing and disk storage to prevent RAM overflow. It orchestrates the entire data preparation pipeline including loading metadata, processing videos in parallel (optional), aggregating results, preparing data for the GRU model, and handling class imbalance.

In [ ]:
# =============================================================================
# BATCH PROCESSING FOR LARGE-SCALE VIDEO PROCESSING
# =============================================================================

def get_temp_dir() -> str:
    """Get or create a temporary directory for intermediate results."""
    temp_dir = os.path.join(tempfile.gettempdir(), 'video_processing_cache')
    os.makedirs(temp_dir, exist_ok=True)
    return temp_dir


def save_batch_to_disk(df: pd.DataFrame, batch_id: int, temp_dir: str = None) -> str:
    """
    Save a processed batch to disk as parquet for memory efficiency.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame to save
    batch_id : int
        Unique batch identifier
    temp_dir : str, optional
        Directory to save files. If None, uses system temp.

    Returns
    -------
    str
        Path to saved file
    """
    if temp_dir is None:
        temp_dir = get_temp_dir()

    file_path = os.path.join(temp_dir, f'batch_{batch_id}.parquet')
    df.to_parquet(file_path, index=False, compression='snappy')
    return file_path


def load_batch_from_disk(file_path: str) -> pd.DataFrame:
    """Load a batch from disk."""
    if os.path.exists(file_path):
        return pd.read_parquet(file_path)
    return pd.DataFrame()


def cleanup_temp_files(temp_dir: str = None, verbose: bool = True) -> None:
    """Remove all temporary batch files."""
    if temp_dir is None:
        temp_dir = get_temp_dir()

    if os.path.exists(temp_dir):
        import shutil
        file_count = len([f for f in os.listdir(temp_dir) if f.endswith('.parquet')])
        shutil.rmtree(temp_dir, ignore_errors=True)
        if verbose:
            print(f"[Cleanup] Removed {file_count} temporary batch files")


def process_single_video_safe(
    video_config: Dict,
    window: int = 105,
    batch_size: int = 15,
    skip_n: int = 2
) -> Optional[pd.DataFrame]:
    """
    Process a single video with error handling and memory optimization.

    Returns None if processing fails or file not found.
    """
    try:
        from time_series.model import DataModeling, optimize_dataframe_memory

        data_model = DataModeling(video_config)
        result = data_model.process(window=window, batch_size=batch_size, skip_n=skip_n)

        if result is not None and not result.empty:
            # Optimize memory before returning
            result = optimize_dataframe_memory(result, verbose=False)
            return result
        return None

    except FileNotFoundError as e:
        print(f"[Skip] File not found: {video_config.get('video_id', 'unknown')}")
        return None
    except Exception as e:
        print(f"[Error] {video_config.get('video_id', 'unknown')}: {str(e)[:100]}")
        return None


def process_video_batch(
    video_configs: List[Dict],
    batch_id: int,
    window: int = 105,
    batch_size: int = 15,
    skip_n: int = 2,
    temp_dir: str = None,
    verbose: bool = True
) -> Tuple[str, int, int]:
    """
    Process a batch of videos and save results to disk.

    Parameters
    ----------
    video_configs : List[Dict]
        List of video configurations to process
    batch_id : int
        Unique batch identifier
    window : int
        Window size for processing
    batch_size : int
        Batch size parameter
    skip_n : int
        Frame skip factor
    temp_dir : str
        Directory for saving intermediate results
    verbose : bool
        Print progress information

    Returns
    -------
    Tuple[str, int, int]
        (file_path, success_count, total_count)
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"Processing Batch {batch_id} ({len(video_configs)} videos)")
        print(f"Memory before: {get_memory_usage_mb():.1f} MB")
        print(f"{'='*60}")

    batch_results = []
    success_count = 0

    for idx, video_config in enumerate(video_configs):
        video_id = video_config.get('video_id', 'unknown')
        lab_id = video_config.get('lab_id', 'unknown')

        if verbose and idx % 10 == 0:
            print(f"  [{idx+1}/{len(video_configs)}] Processing {lab_id}/{video_id}...")

        try:
            data_model = DataModeling(video_config)
            result = data_model.process(window=window, batch_size=batch_size, skip_n=skip_n)

            if result is not None and not result.empty:
                # Optimize memory
                result = optimize_dataframe_memory(result, verbose=False)
                batch_results.append(result)
                success_count += 1

            # Clear intermediate variables
            del data_model
            if 'result' in dir():
                del result

        except FileNotFoundError:
            if verbose:
                print(f"    [Skip] File not found: {video_id}")
            continue
        except Exception as e:
            if verbose:
                print(f"    [Error] {video_id}: {str(e)[:80]}")
            continue

        # Periodic garbage collection every 20 videos
        if idx % 20 == 0:
            gc.collect()

    # Combine batch results
    if batch_results:
        batch_df = pd.concat(batch_results, ignore_index=True)
        batch_df = optimize_dataframe_memory(batch_df, verbose=False)

        # Save to disk
        file_path = save_batch_to_disk(batch_df, batch_id, temp_dir)

        if verbose:
            print(f"  Batch {batch_id}: {success_count}/{len(video_configs)} videos -> {len(batch_df)} rows")
            print(f"  Saved to: {file_path}")

        # Clear memory
        del batch_results
        del batch_df
        gc.collect()

        if verbose:
            print(f"  Memory after cleanup: {get_memory_usage_mb():.1f} MB")

        return file_path, success_count, len(video_configs)
    else:
        if verbose:
            print(f"  Batch {batch_id}: No successful results")
        return None, 0, len(video_configs)


def aggregate_batches_from_disk(
    batch_files: List[str],
    verbose: bool = True
) -> pd.DataFrame:
    """
    Aggregate all batch files from disk into a single DataFrame.
    Uses streaming to minimize memory usage.

    Parameters
    ----------
    batch_files : List[str]
        List of paths to batch parquet files
    verbose : bool
        Print progress information

    Returns
    -------
    pd.DataFrame
        Aggregated DataFrame with globally unique window IDs
    """
    if not batch_files:
        return pd.DataFrame()

    # Filter out None entries
    batch_files = [f for f in batch_files if f is not None and os.path.exists(f)]

    if not batch_files:
        return pd.DataFrame()

    if verbose:
        print(f"\n{'='*60}")
        print(f"Aggregating {len(batch_files)} batch files from disk")
        print(f"{'='*60}")

    accumulated_dfs = []
    global_window_offset = 0
    total_rows = 0

    for idx, file_path in enumerate(batch_files):
        if verbose:
            print(f"  Loading batch {idx+1}/{len(batch_files)}...")

        df = pd.read_parquet(file_path)

        if df.empty:
            continue

        # Reassign window IDs to be globally unique
        if 'window_id' in df.columns:
            local_window_ids = df['window_id'].unique()
            window_id_map = {old_id: global_window_offset + i
                           for i, old_id in enumerate(local_window_ids)}
            df['window_id'] = df['window_id'].map(window_id_map)
            global_window_offset += len(local_window_ids)

        # Optimize memory
        df = optimize_dataframe_memory(df, verbose=False)
        accumulated_dfs.append(df)
        total_rows += len(df)

        if verbose:
            print(f"    Loaded {len(df)} rows (total: {total_rows})")

        # Periodic memory cleanup
        if idx % 5 == 0:
            gc.collect()

    if not accumulated_dfs:
        return pd.DataFrame()

    if verbose:
        print(f"  Concatenating {len(accumulated_dfs)} DataFrames...")

    # Final concatenation
    master_df = pd.concat(accumulated_dfs, ignore_index=True)
    master_df = optimize_dataframe_memory(master_df, verbose=verbose)

    # Clear intermediate list
    del accumulated_dfs
    gc.collect()

    if verbose:
        print(f"  Final dataset: {len(master_df)} rows, {len(master_df.columns)} columns")
        print(f"  Memory usage: {master_df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

    return master_df


def main_optimized(
    balance_strategy: str = 'weights',
    resample_config: Optional[dict] = None,
    weight_config: Optional[dict] = None,
    batch_size_videos: int = 50,
    window: int = 105,
    processing_batch_size: int = 15,
    skip_n: int = 2,
    max_workers: int = None,
    use_parallel: bool = False,
    cleanup_temp: bool = True,
    verbose: bool = True,
    max_windows: int = None
):
    """
    Memory-optimized main function for processing 1000+ videos.

    Uses batch processing with disk-based intermediate storage to prevent RAM overflow.

    ⚠️  MEMORY WARNING: For systems with limited RAM (< 32GB), use max_windows parameter
        to limit dataset size. Recommended: max_windows=100000 for 16GB RAM.

    Parameters
    ----------
    balance_strategy : str
        Strategy for handling imbalanced classes
    resample_config : dict, optional
        Configuration for resampling
    weight_config : dict, optional
        Configuration for weight computation
    batch_size_videos : int
        Number of videos to process per batch before saving to disk (default: 50)
    window : int
        Window size for feature extraction (default: 105)
    processing_batch_size : int
        Batch size parameter for processing (default: 15)
    skip_n : int
        Frame skip factor (default: 2)
    max_workers : int, optional
        Max parallel workers. If None, uses cpu_count() // 2
    use_parallel : bool
        Whether to use parallel processing (default: False for memory safety)
    cleanup_temp : bool
        Whether to cleanup temporary files after aggregation (default: True)
    verbose : bool
        Print detailed progress information

    Returns
    -------
    tuple
        (X_tensor, y_tensor, label_encoder, scaler, master_training_set, class_weights)
    """
    import time
    start_time = time.time()

    if verbose:
        print("="*70)
        print("MEMORY-OPTIMIZED VIDEO PROCESSING PIPELINE")
        print("="*70)
        print(f"Configuration:")
        print(f"  - Videos per batch: {batch_size_videos}")
        print(f"  - Window size: {window}")
        print(f"  - Parallel processing: {use_parallel}")
        print(f"  - Initial memory: {get_memory_usage_mb():.1f} MB")
        print("="*70)

    # Setup temp directory
    temp_dir = get_temp_dir()
    if verbose:
        print(f"Temp directory: {temp_dir}")

    # Load video metadata
    try:
        videos_data = pd.read_csv('/content/drive/MyDrive/MABe-mouse-behavior-detection/train.csv')
        videos_data = videos_data.fillna(value='unknown')

        # Exclude specific labs that cause issues
        videos_data = videos_data[videos_data['lab_id'] != 'MABe22_keypoints']
        videos_data = videos_data[videos_data['lab_id'] != 'MABe22_movies']
        print(f"  Excluded labs: MABe22_keypoints, MABe22_movies")
    except FileNotFoundError as e:
        print(f"[Error] train.csv file not found: {e}")
        return None, None, None, None, pd.DataFrame(), None
    except Exception as e:
        print(f"[Error] Error reading train.csv: {e}")
        return None, None, None, None, pd.DataFrame(), None

    # Get all video configurations as a list
    data_config = list(iterate_video_metadata(videos_data))
    total_videos = len(data_config)

    if verbose:
        print(f"\nFound {total_videos} videos to process")

    # Split into batches
    video_batches = list(chunk_list(data_config, batch_size_videos))
    num_batches = len(video_batches)

    if verbose:
        print(f"Split into {num_batches} batches of ~{batch_size_videos} videos each\n")

    # Process each batch and save to disk
    batch_files = []
    total_success = 0
    total_processed = 0

    for batch_id, video_batch in enumerate(video_batches):
        file_path, success, processed = process_video_batch(
            video_configs=video_batch,
            batch_id=batch_id,
            window=window,
            batch_size=processing_batch_size,
            skip_n=skip_n,
            temp_dir=temp_dir,
            verbose=verbose
        )

        if file_path:
            batch_files.append(file_path)

        total_success += success
        total_processed += processed

        # Force memory cleanup between batches
        clear_memory(verbose=verbose)

        if verbose:
            elapsed = time.time() - start_time
            videos_per_sec = total_processed / elapsed if elapsed > 0 else 0
            remaining = (total_videos - total_processed) / videos_per_sec if videos_per_sec > 0 else 0
            print(f"\nProgress: {total_processed}/{total_videos} videos ({100*total_processed/total_videos:.1f}%)")
            print(f"Success rate: {total_success}/{total_processed} ({100*total_success/total_processed:.1f}%)")
            print(f"Speed: {videos_per_sec:.2f} videos/sec | ETA: {remaining/60:.1f} min")

    if verbose:
        print(f"\n{'='*70}")
        print(f"BATCH PROCESSING COMPLETE")
        print(f"  Total videos: {total_videos}")
        print(f"  Successful: {total_success}")
        print(f"  Batch files created: {len(batch_files)}")
        print(f"{'='*70}")

    # Aggregate all batches from disk
    if not batch_files:
        print("[Error] No successful batches to aggregate")
        return None, None, None, None, pd.DataFrame(), None

    master_training_set = aggregate_batches_from_disk(batch_files, verbose=verbose)

    if master_training_set.empty:
        print("[Error] Aggregation resulted in empty dataset")
        return None, None, None, None, pd.DataFrame(), None

    # Cleanup temp files
    if cleanup_temp:
        cleanup_temp_files(temp_dir, verbose=verbose)

    # Prepare data for GRU
    if verbose:
        print(f"\nPreparing data for GRU model...")
        print(f"  Dataset: {len(master_training_set):,} rows, {master_training_set['window_id'].nunique():,} windows")

    # Subsample if max_windows is set (to prevent RAM overflow)
    if max_windows is not None:
        n_windows = master_training_set['window_id'].nunique()
        if n_windows > max_windows:
            if verbose:
                print(f"\n⚠️  Subsampling to prevent memory overflow...")
                print(f"  Current: {n_windows:,} windows -> Target: {max_windows:,} windows")
            master_training_set = subsample_windows(
                master_training_set,
                max_windows=max_windows,
                stratified=True
            )
            gc.collect()

    X_tensor, y_tensor, label_encoder, scaler = prepare_data_for_gru(
        master_training_set, FEATURES, target_window_size=window
    )

    # Clear master_training_set to free memory (data is now in tensors)
    del master_training_set
    gc.collect()

    # Handle imbalanced data
    if verbose:
        print(f"Handling imbalanced data with strategy: {balance_strategy}")

    X_balanced, y_balanced, class_weights = handle_imbalanced_data(
        X_tensor, y_tensor,
        strategy=balance_strategy,
        resample_config=resample_config,
        weight_config=weight_config
    )

    # Final stats
    elapsed = time.time() - start_time
    if verbose:
        print(f"\n{'='*70}")
        print(f"PIPELINE COMPLETE")
        print(f"  Total time: {elapsed/60:.1f} minutes")
        print(f"  Final dataset shape: X={X_balanced.shape}, y={y_balanced.shape}")
        print(f"  Final memory: {get_memory_usage_mb():.1f} MB")
        print(f"{'='*70}")

    # Return None for master_training_set since it was cleared to save memory
    # Data is now in X_balanced and y_balanced tensors
    return X_balanced, y_balanced, label_encoder, scaler, None, class_weights

This cell defines core functional programming utilities:
- `pipe()`: A function composition utility that applies a series of functions to a value from left to right.
- `curry()`: A decorator that transforms a function into its curried form, allowing for partial application of arguments.
- `safe_operation()`: A decorator that wraps a function to handle exceptions gracefully, returning a default value if an error occurs.

In [ ]:
# =============================================================================
# CORE FUNCTIONAL UTILITIES
# =============================================================================

def pipe(*functions: Callable) -> Callable:
    """Compose functions left-to-right (pipe operator)."""
    return lambda x: reduce(lambda acc, f: f(acc), functions, x)


def curry(func: Callable) -> Callable:
    """Convert a function into curried form for partial application."""
    sig = inspect.signature(func)
    num_required = sum(
        1 for p in sig.parameters.values()
        if p.default is inspect._empty
        and p.kind in (p.POSITIONAL_ONLY, p.POSITIONAL_OR_KEYWORD)
    )

    @wraps(func)
    def curried(*args, **kwargs):
        if len(args) + len(kwargs) >= num_required:
            return func(*args, **kwargs)
        return curry(partial(func, *args, **kwargs))

    return curried


def safe_operation(func: Callable, default_value: Any = None) -> Callable:
    """Wrap function to handle errors gracefully."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            print(f"Warning: {func.__name__} failed with {e}, returning default")
            return default_value if default_value is not None else (args[0] if args else None)
    return wrapper

This cell contains several data processing and feature engineering functions, all designed to be memory-efficient and robust:
- `iterate_video_metadata()`: A curried generator function that iterates through a DataFrame of video metadata, yielding a dictionary of properties for each unique video. It validates required columns and efficiently extracts unique video configurations.
- `read_file()`: A curried function to safely read CSV or Parquet files from a specified folder, automatically detecting file type and handling missing files by returning an empty DataFrame.
- `insert_other_behaviour_annotations()`: A curried function that analyzes annotation data, identifies gaps between consecutive actions for the same agent, and inserts 'other_behaviour' annotations for sufficiently long gaps. This helps in capturing non-specific behaviors.
- `organize_anot_data_action_order()`: A curried function that expands annotations into fixed-size sequence windows based on either 'centered' or 'sliding' modes. It generates a DataFrame with detailed window information.
- `interpolation_frames()`: A curried function to resample and interpolate numeric columns in tracking data to a target frames-per-second (`t_fps`). It handles missing time data, duplicate timestamps, and applies different fill methods for out-of-range values. This function also includes `pd.DataFrame` imports from `typing`.
- `extracting_bodyparts_coordinates()`: A curried function that pivots body part coordinates from a long format to a wide format, creating columns like `x_head`, `y_nose`. It then fills missing data per mouse using optimized forward and backward fill methods and dynamically extracts unique body parts.
- `extracting_features()`: A curried function that calculates various kinematic and spatial features from processed tracking data. This includes computing face and body centers, distances between body parts, body and face orientations, speeds, accelerations, angular velocities, and distances to walls and the arena center. It uses vectorized operations for efficiency.
- `build_training_rows_from_annotations_optimized()`: A highly optimized function for building a labeled timeline from feature data and annotations. It uses integer-based indexing and numpy arrays for speed, efficiently assigning labels (e.g., 'start_action', 'action', 'end_action', 'other') to each frame.
- `slice_windows_iterative_pointer()`: An iterative builder that extracts fixed-size windows from the labeled timeline, focusing on `start_` and `end_` action tags. It uses a pointer system to process windows sequentially and applies padding logic with 'other' labels.
- `sample_background_windows()`: Extracts 'other' (background) windows from the gaps between actions. It prioritizes

In [ ]:
@curry
def iterate_video_metadata(
    metadata_df: pd.DataFrame,
    lab_col: str = "lab_id",
    video_col: str = "video_id",
    # The additional columns you requested
    extra_cols: List[str] = [
        'frames_per_second', 'video_duration_sec', 'pix_per_cm_approx',
        'video_width_pix', 'video_height_pix', 'arena_width_cm', 'arena_height_cm',
        'arena_shape', 'arena_type'
    ]
) -> Generator[Dict[str, Any], None, None]:
    """
    Iterates through unique videos in metadata and returns a dictionary of their properties.

    Parameters
    ----------
    metadata_df : pd.DataFrame
        DataFrame containing lab_id, video_id, and the extra metadata columns.
    lab_col : str
        Name of the lab ID column.
    video_col : str
        Name of the video ID column.
    extra_cols : List[str]
        List of additional column names to retrieve.

    Yields
    ------
    Dict[str, Any]
        A dictionary for each unique video containing:
        {
            'lab_id': ...,
            'video_id': ...,
            'frames_per_second': ...,
            ...
        }
    """

    # 1. DEFINE REQUIRED COLUMNS
    # We need the ID columns + the extra requested columns
    required_cols = [lab_col, video_col] + extra_cols

    # 2. VALIDATE COLUMNS EXIST
    missing_cols = [c for c in required_cols if c not in metadata_df.columns]
    if missing_cols:
        raise ValueError(f"Metadata DataFrame is missing columns: {missing_cols}")

    # 3. EXTRACT UNIQUE ROWS
    # We drop duplicates based on the Lab/Video ID to ensure we get one info block per video.
    # We keep the first occurrence of metadata found for that video.
    unique_videos = metadata_df[required_cols].drop_duplicates(subset=[lab_col, video_col])

    # 4. ITERATE AND YIELD DICTIONARIES
    # Use itertuples for better performance than iterrows
    for row in unique_videos.itertuples(index=False):
        # Convert namedtuple to dictionary
        video_info = {col: getattr(row, col) for col in required_cols}
        yield video_info
@curry
def read_file(folder_name: str, lab_id: str, file_id: str, file_type: str = None) -> pd.DataFrame:
    """
    Read a file (CSV or Parquet) from the specified folder.

    Parameters
    ----------
    folder_name : str
        Name of the folder containing the file
    lab_id : str
        Lab identifier (subfolder)
    file_id : str
        File identifier (without extension)
    file_type : str, optional
        File type ('parquet' or 'csv'). If None, will auto-detect.

    Returns
    -------
    pd.DataFrame
        Loaded DataFrame, or empty DataFrame if file not found
    """
    if not folder_name or not lab_id or not file_id:
        raise ValueError("folder_name, lab_id, and file_id must be non-empty strings")

    base_path = Path("/content/drive/MyDrive/MABe-mouse-behavior-detection") / folder_name / lab_id

    if file_type is None:
        for ext in ["parquet", "csv"]:
            file_path = base_path / f"{file_id}.{ext}"
            if file_path.exists():
                file_type = ext
                break
        if file_type is None:
            print(f"[Warning] No CSV or Parquet file found for: {file_id} in {base_path}. Returning empty DataFrame.")
            return pd.DataFrame()

    file_path = base_path / f"{file_id}.{file_type}"

    if not file_path.exists():
        print(f"[Warning] File not found: {file_path}. Returning empty DataFrame.")
        return pd.DataFrame()

    try:
        if file_type == 'parquet':
            return pd.read_parquet(file_path)
        elif file_type == 'csv':
            return pd.read_csv(file_path)
        else:
            raise TypeError(f"Unsupported file type: {file_type}")
    except FileNotFoundError as e:
        print(f"[Warning] File not found: {file_path}. Returning empty DataFrame.")
        return pd.DataFrame()
    except Exception as e:
        print(f"[Warning] Error reading file {file_path}: {e}. Returning empty DataFrame.")
        return pd.DataFrame()
@curry
def insert_other_behaviour_annotations(
    anot_df: pd.DataFrame,
    safe_distance: int = 10,
    min_frames: int = 5,
    agent_col: str = "agent_id",
    target_col: str = "target_id",
    start_col: str = "start_frame",
    stop_col: str = "stop_frame",
    action_col: str = "action",
    new_action_name: str = "other"
) -> pd.DataFrame:
    """
    Find gaps between consecutive annotations for the same agent and add
    'other_behaviour' annotations for sufficiently long gaps.

    Parameters
    ----------
    anot_df : pd.DataFrame
        Annotations DataFrame. Must contain agent_col, start_col, stop_col.
    safe_distance : int
        Number of frames to leave as margin before/after existing annotations.
    min_frames : int
        Minimum number of frames required to create an 'other_behaviour' segment.
    column names: customizable in case your columns differ.

    Returns
    -------
    pd.DataFrame
        New DataFrame = original + inserted other_behaviour rows, sorted by agent & start_frame.
    """
    # basic checks
    if start_col not in anot_df.columns or stop_col not in anot_df.columns or agent_col not in anot_df.columns:
        raise ValueError(f"Input DataFrame must contain '{agent_col}', '{start_col}', and '{stop_col}' columns.")

    # work on a sorted copy
    df = anot_df.copy()
    df = df.sort_values([agent_col, start_col]).reset_index(drop=True)

    # compute next start within each agent group using groupby + shift
    df['next_start'] = df.groupby(agent_col)[start_col].shift(-1)

    # compute candidate gap bounds
    df['gap_start'] = df[stop_col] + safe_distance
    df['gap_stop'] = df['next_start'] - safe_distance

    # compute number of frames available in the gap
    df['gap_frames'] = df['gap_stop'] - df['gap_start']

    # select valid gap rows: next_start exists, gap_frames >= min_frames
    mask_valid = df['next_start'].notna() & (df['gap_frames'] >= min_frames)

    # if nothing to add, return original (sorted) anot_df copy
    if not mask_valid.any():
        # clean temporary cols and return original sorted
        df = df.drop(columns=['next_start','gap_start','gap_stop','gap_frames'])
        return df.sort_values([agent_col, start_col]).reset_index(drop=True)

    # Build new rows DataFrame
    new_rows = df.loc[mask_valid, [agent_col, target_col]].copy()
    # If target_col not present, fill with NaN
    if target_col not in new_rows.columns:
        new_rows[target_col] = np.nan

    new_rows = new_rows.reset_index(drop=True)
    new_rows[start_col] = df.loc[mask_valid, 'gap_start'].astype(int).values
    new_rows[stop_col] = df.loc[mask_valid, 'gap_stop'].astype(int).values
    new_rows[action_col] = new_action_name
    # other columns from original anot_df not included above: set to NaN
    for c in anot_df.columns:
        if c not in new_rows.columns:
            new_rows[c] = np.nan

    # keep same column order as original
    new_rows = new_rows[anot_df.columns]

    # concat and sort
    out = pd.concat([anot_df, new_rows], ignore_index=True, sort=False)
    out = out.sort_values([agent_col, start_col]).reset_index(drop=True)

    # drop temporary columns from original df if present
    out = out.loc[:, ~out.columns.duplicated()]
    return out
from typing import Optional, Iterable
import numpy as np
import pandas as pd

@curry
def organize_anot_data_action_order(
    anot_df: pd.DataFrame,
    batch_size: int = 30,
    action_col: str = "action",
    agent_col: str = "agent_id",
    target_col: str = "target_id",
    start_col: str = "start_frame",
    stop_col: str = 'stop_frame',
    mode: str = "centered",      # "centered" or "sliding"
    allow_partial: bool = False, # include last partial window if sliding and leftover < batch_size
    clamp_min_frame: int = 0,
    clamp_max_frame: Optional[int] = None
) -> pd.DataFrame:
    """
    Expand annotations into fixed-size sequence windows.

    Parameters
    ----------
    anot_df : DataFrame
      Input annotations (each row one action) with start_col/stop_col inclusive or exclusive depending on your convention.
    batch_size : int
      Number of frames per sequence window.
    mode : "centered" or "sliding"
      - "centered": place n_batches = floor(length / batch_size) windows centered on annotation middle,
      - "sliding" : place windows starting from start_frame stepping by batch_size (optionally include last partial).
    allow_partial : bool
      If True and mode == "sliding", include a final partial window shorter than batch_size.
    clamp_min_frame / clamp_max_frame : ints
      clamp the window starts/ends to these frame bounds if provided (clamp_max_frame is optional).

    Returns
    -------
    DataFrame with one row per window and columns:
      [agent_col, target_col, action_col, anno_start, anno_stop,
       seq_index, window_start_frame, window_end_frame, window_length, middle_frame]
    """
    df = anot_df.copy().reset_index(drop=True)
    required = {start_col, stop_col}
    for c in required:
        if c not in df.columns:
            raise ValueError(f"Input DataFrame missing required column: {c}")

    rows = []
    # treat start/stop as inclusive by default (document and adapt if your schema is different)
    df[start_col] = df[start_col].astype(int)
    df[stop_col]  = df[stop_col].astype(int)

    for _, r in df.sort_values([agent_col, start_col]).iterrows():
        a_start = int(r[start_col])
        a_stop = int(r[stop_col])
        # inclusive length
        length = a_stop - a_start + 1

        if length <= 0:
            # skip or continue depending on policy
            continue

        middle = a_start + length // 2

        if mode == "sliding":
            # starts at a_start, step by batch_size
            starts = list(range(a_start, a_stop - batch_size + 1, batch_size))
            if allow_partial:
                last_start = a_stop - batch_size + 1
                if last_start <= a_stop and (len(starts) == 0 or starts[-1] != last_start):
                    starts.append(last_start)
            n_batches = len(starts)
            for i, s in enumerate(starts):
                e = min(s + batch_size - 1, a_stop)
                # clamp globally if requested
                s_clamped = max(clamp_min_frame, s)
                e_clamped = e if clamp_max_frame is None else min(e, clamp_max_frame)
                rows.append({
                    agent_col: r.get(agent_col, None),
                    target_col: r.get(target_col, None),
                    action_col: r.get(action_col, None),
                    "anno_start": a_start,
                    "anno_stop" : a_stop,
                    "seq_index": i,
                    "window_start_frame": int(s_clamped),
                    "window_end_frame": int(e_clamped),
                    "window_length": int(e_clamped - s_clamped + 1),
                    "middle_frame": int(middle)
                })
        elif mode == "centered":
            n_batches = length // batch_size
            if n_batches <= 0:
                # fallback: create a single window covering the annotation (or skip)
                s = a_start
                e = a_stop
                s_clamped = max(clamp_min_frame, s)
                e_clamped = e if clamp_max_frame is None else min(e, clamp_max_frame)
                rows.append({
                    agent_col: r.get(agent_col, None),
                    target_col: r.get(target_col, None),
                    action_col: r.get(action_col, None),
                    "anno_start": a_start,
                    "anno_stop" : a_stop,
                    "seq_index": 0,
                    "window_start_frame": int(s_clamped),
                    "window_end_frame": int(e_clamped),
                    "window_length": int(e_clamped - s_clamped + 1),
                    "middle_frame": int(middle)
                })
            else:
                total_len = n_batches * batch_size
                first_start = middle - (total_len // 2)
                # ensure first_start does not exceed annotation bounds
                first_start = max(first_start, a_start)
                # if the block exceeds a_stop, shift left
                if first_start + total_len - 1 > a_stop:
                    first_start = a_stop - total_len + 1
                for i in range(n_batches):
                    s = first_start + i * batch_size
                    e = s + batch_size - 1
                    s_clamped = max(clamp_min_frame, s)
                    e_clamped = e if clamp_max_frame is None else min(e, clamp_max_frame)
                    rows.append({
                        agent_col: r.get(agent_col, None),
                        target_col: r.get(target_col, None),
                        action_col: r.get(action_col, None),
                        "anno_start": a_start,
                        "anno_stop" : a_stop,
                        "seq_index": i,
                        "window_start_frame": int(s_clamped),
                        "window_end_frame": int(e_clamped),
                        "window_length": int(e_clamped - s_clamped + 1),
                        "middle_frame": int(middle)
                    })
        else:
            raise ValueError("mode must be 'centered' or 'sliding'")

    out = pd.DataFrame(rows)
    # keep consistent dtypes
    if not out.empty:
        int_cols = ["anno_start", "anno_stop", "seq_index", "window_start_frame", "window_end_frame", "window_length", "middle_frame"]
        for c in int_cols:
            if c in out.columns:
                out[c] = out[c].astype(int)
    return out
@curry
def interpolation_frames(
    File: pd.DataFrame,
    fps: int,
    t_fps: int,
    numeric_cols: Optional[List[str]] = None,
    meta_cols: Optional[List[str]] = None,
    fill_method: str = 'nan',
    constant_fill_value: float = np.nan,
) -> pd.DataFrame:
    """
    Resample/interpolate numeric columns to a target frames-per-second (t_fps).

    Parameters
    ----------
    File : pd.DataFrame
        Input dataframe with at least ['mouse_id','bodypart','video_frame'] or 'time_sec'.
    fps : int
        Original frames per second used to compute time_sec if missing.
    t_fps : int
        Target frames per second for resampling (used for target times and video_frame).
    numeric_cols : list[str] or None
        Numeric columns to interpolate. If None, all numeric columns excluding index/meta/id/time are used.
    meta_cols : list[str] or None
        Metadata columns to copy over from the first row of each group (e.g., ['lab_id','video_id']).
        If None, defaults to ['lab_id','video_id','arena_type','arena_shape'] where present.
    fill_method : {'nan','nearest','constant'}
        How to fill times outside the available range when interpolation would require extrapolation:
         - 'nan': keep NaN (default)
         - 'nearest': use the nearest endpoint value (forward/backward fill)
         - 'constant': fill with `constant_fill_value`
    constant_fill_value : float
        Value used when fill_method == 'constant'.

    Returns
    -------
    pd.DataFrame
        Interpolated DataFrame with columns:
        meta_cols (if present), mouse_id, bodypart, time_sec, video_frame, <numeric_cols...>, ...
    """
    # Copy to avoid side effects
    df = File.copy()

    # Ensure time_sec exists
    if 'time_sec' not in df.columns:
        if 'video_frame' not in df.columns:
            raise KeyError("Input must contain 'video_frame' or 'time_sec'.")
        df['time_sec'] = df['video_frame'] / float(fps)

    # Default meta cols
    if meta_cols is None:
        meta_cols = ['lab_id', 'video_id', 'arena_type', 'arena_shape']

    # Determine numeric columns to interpolate
    if numeric_cols is None:
        # take numeric dtypes except id/time columns
        exclude = {'mouse_id', 'bodypart', 'video_frame', 'time_sec'}
        if isinstance(df.index, pd.MultiIndex):
            exclude.update(df.index.names or [])
        numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]

    # fallback ensure x and y present in numeric_cols if exist in df
    for c in ['x', 'y']:
        if c in df.columns and c not in numeric_cols:
            numeric_cols.append(c)

    # Prepare container for results
    out_groups = []

    # Pre-calc step size for target times
    step = 1.0 / float(t_fps)

    # Group by mouse_id & bodypart
    group_iter = df.groupby(['mouse_id', 'bodypart'], sort=False)

    for (mouse_id, bodypart), g in group_iter:
        g = g.sort_values('time_sec', kind='mergesort').reset_index(drop=True)
        t_min, t_max = g['time_sec'].min(), g['time_sec'].max()

        # Build target times (include end with small epsilon)
        if np.isclose(t_min, t_max):
            target_times = np.array([t_min])
        else:
            target_times = np.arange(t_min, t_max + 1e-10 + step/2, step)  # include last within tolerance

        new_group = pd.DataFrame({'time_sec': target_times})

        # For each numeric column, interpolate with handling NaNs, duplicates, single points
        for col in numeric_cols:
            if col not in g.columns:
                new_group[col] = np.nan
                continue

            vals = g[col].to_numpy(dtype=float)
            times = g['time_sec'].to_numpy(dtype=float)

            # Mask invalid entries
            valid = ~np.isnan(vals)
            if valid.sum() == 0:
                new_vals = np.full_like(target_times, np.nan, dtype=float)
            elif valid.sum() == 1:
                # Single valid value -> fill constant
                new_vals = np.full_like(target_times, vals[valid][0], dtype=float)
            else:
                t_valid = times[valid]
                v_valid = vals[valid]

                # Collapse duplicate timestamps by averaging values at same time
                if np.unique(t_valid).size != t_valid.size:
                    tmp = pd.DataFrame({'t': t_valid, 'v': v_valid})
                    tmp = tmp.groupby('t', as_index=False)['v'].mean()
                    t_valid = tmp['t'].to_numpy()
                    v_valid = tmp['v'].to_numpy()

                # Ensure ascending times required by np.interp
                sort_idx = np.argsort(t_valid)
                t_valid = t_valid[sort_idx]
                v_valid = v_valid[sort_idx]

                # Use numpy.interp for fast linear interpolation (extrapolation uses endpoint values)
                new_vals = np.interp(target_times, t_valid, v_valid)

                # Handle fill_method: set out-of-range samples to NaN or nearest or constant
                # Determine mask where target_times < min or > max
                mask_left = target_times < t_valid[0]
                mask_right = target_times > t_valid[-1]
                out_of_range = mask_left | mask_right

                if fill_method == 'nan':
                    new_vals[out_of_range] = np.nan
                elif fill_method == 'nearest':
                    # left: use first valid, right: use last valid (np.interp already gave endpoint values)
                    # so nothing to do (np.interp already provides nearest endpoint for extrapolation)
                    pass
                elif fill_method == 'constant':
                    new_vals[mask_left] = constant_fill_value
                    new_vals[mask_right] = constant_fill_value
                else:
                    raise ValueError("fill_method must be one of {'nan','nearest','constant'}")

            new_group[col] = new_vals

        # IDs and video_frame according to target fps
        new_group['mouse_id'] = mouse_id
        new_group['bodypart'] = bodypart
        new_group['video_frame'] = (new_group['time_sec'] * float(t_fps)).round().astype(int)

        # Copy metadata columns if present
        for m in meta_cols:
            if m in g.columns:
                new_group[m] = g.iloc[0][m]

        out_groups.append(new_group)

    # If no groups, return empty frame with sensible columns
    if not out_groups:
        cols = list(set(['mouse_id', 'bodypart', 'time_sec', 'video_frame'] + numeric_cols + meta_cols))
        return pd.DataFrame(columns=cols)

    res = pd.concat(out_groups, ignore_index=True, sort=False)

    # Reorder columns: meta first, then ids/time/frame, then numeric cols, then any others
    pref = [c for c in (meta_cols + ['mouse_id', 'bodypart', 'time_sec', 'video_frame']) if c in res.columns]
    numeric_present = [c for c in numeric_cols if c in res.columns]
    rest = [c for c in res.columns if c not in pref + numeric_present]
    res = res[pref + numeric_present + rest]

    return res
import pandas as pd
from typing import Tuple, List

This cell contains functions to aggregate data and preprocess it for GRU model training:
- `aggregate_multi_video_datasets()`: Merges processed datasets from multiple videos or labs into a single master DataFrame, ensuring globally unique `window_id`s and unifying action categories. It also stamps `lab_id` and `video_id` metadata onto each row.
- `skip_frames()`: A curried function to downsample a DataFrame by skipping frames based on a specified `skip_factor`. This can reduce the dataset size for memory efficiency.
- `FEATURES`: A global list defining the names of feature columns used for model training.
- `normalize_action_labels()`: A curried function to clean up action labels by removing 'start_' and 'end_' prefixes, merging related categories (e.g., 'start_attack', 'end_attack', 'attack' all become 'attack').
- `prepare_data_for_gru()`: A memory-optimized function that takes a DataFrame of features and labels, sorts it, handles missing values using various strategies (e.g., interpolate, ffill, mean, zero), pads or truncates windows to a fixed `target_window_size`, normalizes features using `StandardScaler`, reshapes the data into a 3D tensor suitable for GRU input (samples, timesteps, features), and encodes labels. It uses `float32` to reduce memory usage.
- `subsample_windows()`: Subsamples windows from a large dataset, either randomly or in a stratified manner, to reduce memory consumption.
- `prepare_data_for_gru_chunked()`: A memory-efficient wrapper around `prepare_data_for_gru` that can subsample windows and optionally save processed tensors to disk.

In [ ]:
@curry
def extracting_bodyparts_coordinates(data: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    """
    Pivot bodypart coordinates into wide format and fill missing data per mouse.

    Optimizations:
    - Uses vectorized groupby().ffill()/bfill() instead of slow .apply().
    - Performs filling while 'mouse_id' is still an index to avoid expensive regrouping.
    - Robust column flattening.
    """
    required_cols = {'mouse_id', 'video_frame', 'time_sec', 'bodypart', 'x', 'y'}
    if not required_cols.issubset(data.columns):
        missing = required_cols - set(data.columns)
        raise ValueError(f"Missing required columns: {missing}")

    # 1. Pivot to wide format
    # Index becomes MultiIndex: (mouse_id, time_sec, video_frame)
    # This automatically sorts the data by these keys.
    new_data = data.pivot(
        index=['mouse_id', 'video_frame'],
        columns='bodypart',
        values=['x', 'y']
    )

    # 2. Flatten MultiIndex columns (e.g., ('x','head') -> 'x_head')
    new_data.columns = [f"{coord}_{part}" for coord, part in new_data.columns]

    # 3. Optimized Gap Filling (Imputation)
    # fast ffill/bfill operating on the specific index level 'mouse_id'
    # This vectorizes the operation across the C backend.
    new_data = new_data.groupby(level='mouse_id').ffill()
    new_data = new_data.groupby(level='mouse_id').bfill()

    # 4. Reset index to return flat DataFrame
    new_data = new_data.reset_index()

    # 5. Extract unique bodyparts dynamically
    # We filter for columns starting with 'x_' to avoid getting duplicates from 'y_'
    unique_bodyparts = sorted([
        col.split('_', 1)[1]
        for col in new_data.columns
        if col.startswith('x_')
    ])

    print(f"Unique bodyparts found: {unique_bodyparts}")

    return new_data, unique_bodyparts

import numpy as np
import pandas as pd
from typing import List, Optional

@curry
def extracting_features(
    new_data: pd.DataFrame,
    fps: int,
    pix_per_cm: Optional[float],
    video_width_pix: float,
    video_height_pix: float,
    unique_bodyparts: List[str]
) -> pd.DataFrame:
    """
    Robust, vectorized feature extraction.

    Optimizations:
    - Eliminates row duplication by avoiding dangerous pd.concat operations on reset indices.
    - Performs calculations directly on the main DataFrame to save memory.
    - Ensures temporal ordering before diff() calculations.
    """
    # 1. PREP: Work on a copy and standardize index
    df = new_data.copy()

    # Ensure no duplicate columns in input (keep first)
    if df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()]

    # Ensure data is sorted for temporal calculations (CRITICAL for .diff() to work)
    if 'mouse_id' in df.columns and 'video_frame' in df.columns:
        df = df.sort_values(by=['mouse_id', 'video_frame'])

    # We will simply add columns to 'df' rather than creating 'computed' and merging.
    # This prevents index misalignment.

    parts = set(unique_bodyparts or [])

    # Helper to get numpy arrays for faster calc (fillna with NaN for safety)
    def get_vec(part, axis):
        col = f'{axis}_{part}'
        if col in df.columns:
            return df[col].to_numpy()
        return np.full(len(df), np.nan)

    # 2. VECTORIZED GEOMETRY CALCULATIONS
    # Fetch coordinates as numpy arrays for speed
    ear_l_x, ear_l_y = get_vec('ear_left', 'x'), get_vec('ear_left', 'y')
    ear_r_x, ear_r_y = get_vec('ear_right', 'x'), get_vec('ear_right', 'y')
    nose_x, nose_y = get_vec('nose', 'x'), get_vec('nose', 'y')
    tail_x, tail_y = get_vec('tail_base', 'x'), get_vec('tail_base', 'y')
    head_x, head_y = get_vec('head', 'x'), get_vec('head', 'y')

    # --- Face Center ---
    # We calculate candidates and then select based on available parts
    # (Doing this via numpy is faster than python if/else if vectors are large)

    # Default: NaNs
    fc_x = np.full(len(df), np.nan)
    fc_y = np.full(len(df), np.nan)

    if {'nose', 'ear_left', 'ear_right'}.issubset(parts):
        fc_x = (nose_x + ear_l_x + ear_r_x) / 3.0
        fc_y = (nose_y + ear_l_y + ear_r_y) / 3.0
    elif {'head', 'ear_left', 'ear_right'}.issubset(parts):
        fc_x = (head_x + ear_l_x + ear_r_x) / 3.0
        fc_y = (head_y + ear_l_y + ear_r_y) / 3.0
    elif 'head' in parts:
        fc_x, fc_y = head_x, head_y

    df['x_face_center'] = fc_x
    df['y_face_center'] = fc_y

    # --- Body Center ---
    # Similar logic for body center
    bc_x = np.full(len(df), np.nan)
    bc_y = np.full(len(df), np.nan)

    lat_l_x, lat_l_y = get_vec('lateral_left', 'x'), get_vec('lateral_left', 'y')
    lat_r_x, lat_r_y = get_vec('lateral_right', 'x'), get_vec('lateral_right', 'y')
    body_c_x, body_c_y = get_vec('body_center', 'x'), get_vec('body_center', 'y')
    hip_l_x, hip_l_y = get_vec('hip_left', 'x'), get_vec('hip_left', 'y')
    hip_r_x, hip_r_y = get_vec('hip_right', 'x'), get_vec('hip_right', 'y')
    neck_x, neck_y = get_vec('neck', 'x'), get_vec('neck', 'y')

    # Logic to choose best body center
    if {'lateral_left', 'lateral_right', 'body_center'}.issubset(parts):
        bc_x = (lat_l_x + lat_r_x + body_c_x) / 3.0
        bc_y = (lat_l_y + lat_r_y + body_c_y) / 3.0
    elif 'body_center' in parts:
        bc_x, bc_y = body_c_x, body_c_y
    elif {'hip_left', 'hip_right', 'neck'}.issubset(parts):
        mid_hips_x = (hip_l_x + hip_r_x) / 2.0
        mid_hips_y = (hip_l_y + hip_r_y) / 2.0
        bc_x = (mid_hips_x + neck_x) / 2.0
        bc_y = (mid_hips_y + neck_y) / 2.0

    df['x_body_center_final'] = bc_x
    df['y_body_center_final'] = bc_y

    # --- Forepaws ---
    fp_l_x, fp_l_y = get_vec('forepaw_left', 'x'), get_vec('forepaw_left', 'y')
    fp_r_x, fp_r_y = get_vec('forepaw_right', 'x'), get_vec('forepaw_right', 'y')

    if {'forepaw_left', 'forepaw_right'}.issubset(parts):
        df['x_forepaw_center'] = (fp_l_x + fp_r_x) / 2.0
        df['y_forepaw_center'] = (fp_l_y + fp_r_y) / 2.0
        df['forepaw_left_right_distance'] = np.sqrt((fp_l_x - fp_r_x)**2 + (fp_l_y - fp_r_y)**2)
    else:
        df['x_forepaw_center'] = np.nan
        df['y_forepaw_center'] = np.nan
        df['forepaw_left_right_distance'] = np.nan

    # 3. ORIENTATIONS AND ANGLES (Vectorized)
    # Ear Center
    ear_c_x = (ear_l_x + ear_r_x) / 2.0
    ear_c_y = (ear_l_y + ear_r_y) / 2.0

    # Body Orientation (Ear Center to Tail Base)
    bo_x = ear_c_x - tail_x
    bo_y = ear_c_y - tail_y
    df['body_length'] = np.sqrt(bo_x**2 + bo_y**2)
    df['body_orientation'] = np.degrees(np.arctan2(bo_y, bo_x))

    # Face Orientation (Ear Center to Nose)
    fo_x = ear_c_x - nose_x
    fo_y = ear_c_y - nose_y
    df['face_length'] = np.sqrt(fo_x**2 + fo_y**2)
    df['face_orientation'] = np.degrees(np.arctan2(fo_y, fo_x))

    # Distances
    df['nose_forepaw_distance'] = np.sqrt((nose_x - df['x_forepaw_center'])**2 + (nose_y - df['y_forepaw_center'])**2)

    # Triangle Area (Vectorized Heron's formula components)
    d_face_tail = np.sqrt((fc_x - tail_x)**2 + (fc_y - tail_y)**2)
    d_tail_body = np.sqrt((tail_x - bc_x)**2 + (tail_y - bc_y)**2)
    d_body_face = np.sqrt((bc_x - fc_x)**2 + (bc_y - fc_y)**2)

    s = (d_face_tail + d_body_face + d_tail_body) / 2.0
    # Use maximum(..., 0) to avoid sqrt of negative small float errors
    df['triangle_body_area'] = np.sqrt(np.maximum(s * (s - d_face_tail) * (s - d_body_face) * (s - d_tail_body), 0.0))

    # 4. MOTION METRICS (Group-wise diffs)
    # Define pixel to cm conversion
    pix2cm = float(pix_per_cm) if (pix_per_cm and pix_per_cm > 0) else np.nan
    frame_time = 1.0 / float(fps) if fps > 0 else 1.0

    # Points we want to track speed for
    track_cols = {
        'face': ('x_face_center', 'y_face_center'),
        'body': ('x_body_center_final', 'y_body_center_final'),
        'tail': ('x_tail_base', 'y_tail_base') # tail_base columns already exist in df
    }

    for name, (cx, cy) in track_cols.items():
        if cx not in df.columns or cy not in df.columns:
            continue

        # Group by mouse to prevent diffing across different mice
        # diff() relies on the sort we did at step 1
        dx = df.groupby('mouse_id')[cx].diff().fillna(0)
        dy = df.groupby('mouse_id')[cy].diff().fillna(0)

        # Calculate speed in pixels
        speed_px = np.sqrt(dx**2 + dy**2) / frame_time
        col_speed_px = f'{name}_center_speed_px_per_s' if name != 'tail' else 'tail_base_speed_px_per_s'
        df[col_speed_px] = speed_px

        # Calculate acceleration
        accel_px = df.groupby('mouse_id')[col_speed_px].diff().fillna(0) / frame_time
        col_accel_px = f'{name}_center_accel_px_per_s2' if name != 'tail' else 'tail_base_accel_px_per_s2'
        df[col_accel_px] = accel_px

        # Calculate CM metrics if applicable
        if not np.isnan(pix2cm):
            col_speed_cm = f'{name}_center_speed_cm_per_s' if name != 'tail' else 'tail_base_speed_cm_per_s'
            col_accel_cm = f'{name}_center_accel_cm_per_s2' if name != 'tail' else 'tail_base_accel_cm_per_s2'

            df[col_speed_cm] = speed_px / pix2cm
            df[col_accel_cm] = accel_px / pix2cm

    # 5. ANGULAR VELOCITY (Circular statistics)
    def calculate_angular_velocity(series_name, out_name):
        if series_name not in df.columns: return

        def calc_circular_diff(x):
            x = x.to_numpy(dtype=float)
            d = np.full_like(x, np.nan)
            if len(x) > 1:
                diff = x[1:] - x[:-1]
                # Normalize to [-180, 180]
                diff = (diff + 180.0) % 360.0 - 180.0
                d[1:] = diff
            return d

        # Calculate diffs
        ang_diff = df.groupby('mouse_id')[series_name].transform(calc_circular_diff)

        # Rate per second
        rate_name = f'{out_name}_turning_rate_deg_per_s'
        df[rate_name] = ang_diff / frame_time
        df[f'{out_name}_angular_speed_deg_per_s'] = np.abs(df[rate_name])

    calculate_angular_velocity('body_orientation', 'body')
    calculate_angular_velocity('face_orientation', 'face')

    # 6. WALL DISTANCE
    bc_x_final = df['x_body_center_final']
    bc_y_final = df['y_body_center_final']

    # Distance to walls (using numpy min for speed)
    dist_x = np.minimum(bc_x_final, video_width_pix - bc_x_final)
    dist_y = np.minimum(bc_y_final, video_height_pix - bc_y_final)

    df['dist_to_wall_x_px'] = dist_x
    df['dist_to_wall_y_px'] = dist_y

    if not np.isnan(pix2cm):
        df['dist_to_nearest_wall_cm'] = np.minimum(dist_x, dist_y) / pix2cm
        df['dist_to_wall_x_cm'] = dist_x / pix2cm
        df['dist_to_wall_y_cm'] = dist_y / pix2cm

    # Distance to center
    center_x, center_y = video_width_pix / 2.0, video_height_pix / 2.0
    dist_center_px = np.sqrt((bc_x_final - center_x)**2 + (bc_y_final - center_y)**2)

    if not np.isnan(pix2cm):
        df['dist_to_center_cm'] = dist_center_px / pix2cm
    else:
        df['dist_to_center_cm'] = np.nan # Or keep px value depending on preference

    # 7. CLEANUP & ORDERING
    # Ensure no duplicate columns exist before returning
    df = df.loc[:, ~df.columns.duplicated()]

    # Define simple column ordering priority
    cols = list(df.columns)
    id_cols = ['mouse_id', 'video_frame', 'time_sec']
    # Move IDs to front, sort the rest alphabetically or by type if desired
    # Here we just ensure IDs are first
    sorted_cols = [c for c in id_cols if c in cols] + [c for c in cols if c not in id_cols]

    return df[sorted_cols]

    def build_training_rows_from_annotations_optimized(
    features_data: pd.DataFrame,
    anot_data: pd.DataFrame,
    window: int,
    fps: int,
    batch_size: int,
    verbose: bool = False,
    # column names
    mouse_col: str = "mouse_id",
    frame_col: str = "video_frame",
    anno_agent_col: str = "agent_id",
    anno_target_col: str = "target_id",
    anno_start_col: str = "anno_start",
    anno_stop_col: str = "anno_stop",
    anno_seq_col: str = "seq_index",
    action_col: str = "action",
) -> pd.DataFrame:
    """
    Highly optimized builder using integer-based indexing and numpy arrays.
    Removes all string concatenation and dictionary lookups from the inner loops.
    """
    # 1. SETUP: Fast Integer Indexing
    if frame_col not in features_data.columns or mouse_col not in features_data.columns:
        raise ValueError(f"features_data must contain '{frame_col}' and '{mouse_col}' columns")

    # Sort features to ensure consistent indexing
    feats = features_data.sort_values([mouse_col, frame_col]).reset_index(drop=True)

    # Map (mouse_id, frame) -> row_index using a MultiIndex
    # This is much faster than string concatenation keys
    feat_index = feats.set_index([mouse_col, frame_col]).index

    # We will populate this array with labels. Initialize with None.
    # Using a numpy object array is faster for string labels than a pandas Series
    labels_arr = np.full(len(feats), None, dtype=object)

    # Pre-calculate max frame to avoid repeated .max() calls
    max_frame = int(feats[frame_col].max())

    # 2. CREATE FAST LOOKUP
    # To avoid repeated index lookups, we can create a dense lookup table if mouse_ids are few
    # OR efficiently slice the sorted dataframe.
    # Since feats is sorted by [mouse, frame], the indices for a specific mouse are contiguous.

    # Create a map: mouse_id -> (start_idx, end_idx, min_frame, max_frame)
    # This allows us to convert global frame numbers to DataFrame indices using simple math.
    mouse_locs = {}

    # Group by mouse to find the index chunks.
    # Since it's already sorted, we can just find the boundaries.
    # (Using groupby on sorted data is fast)
    for m_id, group in feats.groupby(mouse_col, sort=False):
        start_idx = group.index[0]
        end_idx = group.index[-1]

        # We assume frames are somewhat continuous or at least monotonic
        # We need the min_frame to calculate offsets
        min_f = group[frame_col].min()
        max_f = group[frame_col].max()

        # We capture the frame values array to handle gaps if frames aren't perfectly sequential
        # This allows: global_idx = searchsorted(frames, target_frame) + start_idx
        frames_vals = group[frame_col].values

        mouse_locs[m_id] = {
            'start_global': start_idx,
            'frames': frames_vals,
            'min_f': min_f,
            'max_f': max_f
        }

    # 3. PROCESS ANNOTATIONS
    anots = anot_data.sort_values([anno_agent_col, anno_start_col]).reset_index(drop=True)
    anots_grouped = anots.groupby(anno_agent_col)

    # Helper: Vectorized label assignment
    def assign_label_range(actor_id, label, start_f, end_f):
        if actor_id is None:
            return

        info = mouse_locs.get(actor_id)
        if info is None:
            return

        # Clamp frames to the actor's actual data range
        s = max(start_f, info['min_f'])
        e = min(end_f, info['max_f'])

        if s > e:
            return

        # Find position within this mouse's block
        # searchsorted is extremely fast (binary search)
        # We look for the start and end insertion points
        f_arr = info['frames']

        # Find indices in the local 'frames' array
        idx_start = np.searchsorted(f_arr, s, side='left')
        idx_end = np.searchsorted(f_arr, e, side='right')

        # Convert to global DataFrame indices
        global_start = info['start_global'] + idx_start
        global_end = info['start_global'] + idx_end

        # Assign label to the slice
        if global_end > global_start:
            labels_arr[global_start:global_end] = label

    # Loop through annotations
    for _, group in anots_grouped:
        group_len = len(group)
        # Pre-fetch columns to avoid .loc overhead inside loop
        starts = group[anno_start_col].values
        stops = group[anno_stop_col].values
        seqs = group[anno_seq_col].values if anno_seq_col in group.columns else np.zeros(group_len)
        agents = group[anno_agent_col].values
        targets = group[anno_target_col].values if anno_target_col in group.columns else [None]*group_len
        actions = group[action_col].values

        for i in range(group_len):
            a_start = int(starts[i])
            a_stop = int(stops[i])
            seq_idx = int(seqs[i])
            agent = agents[i]
            target = targets[i]
            act_name = actions[i]

            # Look ahead for next sequence
            next_seq_idx = int(seqs[i+1]) if (i + 1 < group_len) else None

            pre_block_start = max(0, a_start - batch_size) if seq_idx == 0 else None
            post_block_end = min(max_frame, a_stop + batch_size) if (next_seq_idx is None or next_seq_idx == 0) else None

            # 1) PRE-BLOCK
            if pre_block_start is not None:
                assign_label_range(agent, f"start_{act_name}", pre_block_start, a_start - 1)
                assign_label_range(target, f"start_co_{act_name}", pre_block_start, a_start - 1)

            # 2) CORE BLOCK
            assign_label_range(agent, act_name, a_start, a_stop)
            assign_label_range(target, f"co_{act_name}", a_start, a_stop)

            # 3) POST-BLOCK
            if post_block_end is not None:
                assign_label_range(agent, f"end_{act_name}", a_stop + 1, post_block_end)
                assign_label_range(target, f"end_co_{act_name}", a_stop + 1, post_block_end)

                # Padding Logic
                total_span_start = pre_block_start if pre_block_start is not None else a_start
                total_len = (post_block_end - total_span_start + 1)

                if total_len < window:
                    pad_needed = window - total_len
                    pad_s = post_block_end + 1
                    pad_e = min(max_frame, post_block_end + pad_needed)
                    assign_label_range(agent, "other", pad_s, pad_e)
                    assign_label_range(target, "other", pad_s, pad_e)

            # 4) Fallback Padding
            core_len = (a_stop - a_start + 1)
            if pre_block_start is None and post_block_end is None and core_len < batch_size:
                need = batch_size - core_len
                pad_s = a_stop + 1
                pad_e = min(max_frame, a_stop + need)
                assign_label_range(agent, "other", pad_s, pad_e)
                assign_label_range(target, "other", pad_s, pad_e)

    # 4. FINALIZE
    # Assign the numpy array to the dataframe
    feats['action'] = labels_arr

    # Filter rows that have an action
    result_df = feats.dropna(subset=['action']).reset_index(drop=True)

    return result_df

@curry
def slice_windows_iterative_pointer(
    labeled_data: pd.DataFrame,
    window_size: int,
    batch_size: int,
    action_col: str = "action",
    verbose: bool = False
) -> pd.DataFrame:
    """
    Iterative builder that enforces strict 'Start -> End' consumption.

    Logic:
    1. Finds a 'start_' tag.
    2. Finds the matching 'end_' tag.
    3. Slices the window [Start : Start + Window].
    4. Pads with 'other' after (End + BatchSize).
    5. *Moves the Search Pointer* to (End Index), ensuring the next
       search begins strictly after the current action finishes.
    """
    if verbose: print("Starting iterative window slicing...")

    # 1. SETUP DATA (Avoid Copying until slice)
    # Reset index for clean 0..N integer indexing
    df = labeled_data.reset_index(drop=True)
    total_frames = len(df)

    # Get actions as efficient numpy string array (Unicode)
    # Handling potential categoricals or objects
    if isinstance(df[action_col].dtype, pd.CategoricalDtype):
        actions_arr = df[action_col].astype(str).to_numpy(dtype='U')
    else:
        actions_arr = df[action_col].astype(str).to_numpy(dtype='U')

    # 2. FIND ALL TAG INDICES
    # We pre-calculate locations of starts and ends to avoid searching strings in the loop
    is_start = np.char.startswith(actions_arr, 'start_')
    all_starts = np.flatnonzero(is_start)

    is_end = np.char.startswith(actions_arr, 'end_')
    all_ends = np.flatnonzero(is_end)

    if len(all_starts) == 0:
        if verbose: print("[Info] No start actions found.")
        return pd.DataFrame()

    collected_windows = []

    # 3. POINTER ITERATION
    # We maintain a pointer to ensure we don't process overlapping starts
    # (though in clean data, starts shouldn't overlap, this enforces your logic)
    current_search_ptr = 0
    window_id_counter = 0

    # Pointers for our pre-calculated arrays
    start_arr_idx = 0
    end_arr_idx = 0

    while start_arr_idx < len(all_starts):
        # A. Find next valid start
        start_idx = all_starts[start_arr_idx]

        # If this start is before our search pointer (i.e., inside previous action), skip it
        if start_idx < current_search_ptr:
            start_arr_idx += 1
            continue

        # B. Find the matching end (first end > start)
        # We advance the end_arr_idx until we find an end after this start
        while end_arr_idx < len(all_ends) and all_ends[end_arr_idx] <= start_idx:
            end_arr_idx += 1

        if end_arr_idx >= len(all_ends):
            # No matching end found for this start (broken annotation at end of file)
            break

        end_idx = all_ends[end_arr_idx]

        # C. EXTRACT WINDOW
        # Calculate window bounds
        win_end_idx = start_idx + window_size

        # Slice the DataFrame (Copy is necessary here to modify it independently)
        # We handle boundary condition if window goes past end of dataframe
        # (iloc handles slice over-bounds gracefully by returning what exists)
        window_df = df.iloc[start_idx : win_end_idx].copy()

        # Determine actual length retrieved (might be < window_size if at EOF)
        actual_len = len(window_df)

        # D. APPLY PADDING LOGIC
        # Padding starts after: (End Index - Start Index) + Batch Size
        # (This is the relative index inside the window)
        relative_end = end_idx - start_idx
        cutoff_idx = relative_end + batch_size

        # Create mask: True for indices > cutoff
        if cutoff_idx < actual_len:
            # We construct the mask only for the valid length
            mask_indices = np.arange(actual_len)
            mask = mask_indices > cutoff_idx

            # Apply 'other' label
            if isinstance(window_df[action_col].dtype, pd.CategoricalDtype):
                if 'other' not in window_df[action_col].cat.categories:
                    window_df[action_col] = window_df[action_col].cat.add_categories('other')

            # Using .loc with the boolean mask
            window_df.loc[mask, action_col] = 'other'

        # If retrieved chunk is smaller than window_size (EOF case),
        # you might want to pad with zeros or drop it.
        # Here we keep it as is, or you can skip:
        if actual_len < window_size:
            # Optional: Extend with empty rows or drop.
            # For now, we accept it as the final partial window.
            pass

        # E. ADD METADATA
        window_df.insert(0, 'window_id', window_id_counter)
        window_df.insert(1, 'local_frame_index', np.arange(actual_len))

        collected_windows.append(window_df)
        window_id_counter += 1

        # F. UPDATE POINTER
        # User requirement: "start search after each end action"
        current_search_ptr = end_idx + 1

        # Move to next start candidate
        start_arr_idx += 1

    # 4. FINALIZE
    if not collected_windows:
        if verbose: print("[Info] No valid windows constructed.")
        return pd.DataFrame()

    final_df = pd.concat(collected_windows, ignore_index=True)

    if verbose:
        print(f"Extraction complete. Created {window_id_counter} windows.")

    return final_df


def sample_background_windows(
    labeled_data: pd.DataFrame,
    window_size: int,
    step_size: int,
    action_col: str = "action",
    feature_col: str = "body_center_speed_px_per_s", # Column to measure "variance" on
    strategy: str = "quantile",  # Options: 'quantile', 'max_variance', 'random_weighted'
    quantile_threshold: float = 0.4, # Drop the bottom 40% of least active windows
    verbose: bool = False
) -> pd.DataFrame:
    """
    Extracts 'other' windows, preferentially selecting those with high variance/activity.

    Parameters
    ----------
    feature_col : str
        The column used to calculate 'interestingness' (e.g., speed, acceleration).
        If None or not found, defaults to calculating variance of x/y coordinates.
    strategy : str
        - 'quantile': Keeps windows above the {quantile_threshold} of activity (removes sleeping).
        - 'max_variance': Sorts and keeps the top N windows.
        - 'random_weighted': Random sampling where higher variance = higher chance.
    """
    if verbose: print(f"Sampling background windows (Strategy: {strategy})...")

    # 1. SETUP & CLEANUP
    df = labeled_data.reset_index(drop=True)

    # Ensure Unicode strings for fast comparison
    if isinstance(df[action_col].dtype, pd.CategoricalDtype):
        actions = df[action_col].astype(str).to_numpy(dtype='U')
    else:
        actions = df[action_col].fillna('other').astype(str).to_numpy(dtype='U')

    # 2. IDENTIFY BACKGROUND BLOCKS
    # Valid = 'other', 'nan', 'None'
    is_background = (actions == 'other') | (actions == 'nan') | (actions == 'None') | (actions == 'co_other')

    if not np.any(is_background):
        if verbose: print("[Info] No background frames found.")
        return pd.DataFrame()

    # Find continuous blocks (Vectorized Run-Length Encoding)
    padded_mask = np.concatenate(([False], is_background, [False]))
    diff = np.diff(padded_mask.astype(int))
    bg_starts = np.flatnonzero(diff == 1)
    bg_ends = np.flatnonzero(diff == -1)

    # 3. GENERATE CANDIDATE INDICES (Strided)
    candidate_starts = []

    for start, end in zip(bg_starts, bg_ends):
        block_len = end - start
        if block_len < window_size: continue

        valid_len = block_len - window_size + 1
        # Use stride to generate candidates
        starts = start + np.arange(0, valid_len, step_size)
        candidate_starts.append(starts)

    if not candidate_starts:
        return pd.DataFrame()

    all_starts = np.concatenate(candidate_starts)

    # 4. CALCULATE VARIANCE SCORE FOR CANDIDATES (The "Different" Logic)
    # We need to extract the feature values for every candidate window to score them.
    # Broadcasting indices: (Num_Candidates, Window_Size)
    window_offsets = np.arange(window_size)
    # Be careful with memory: if all_starts is huge, do this in chunks or ensure RAM is enough.
    # For <1M candidates, this matrix is fine.
    candidate_indices = all_starts[:, None] + window_offsets

    # Select Feature for Scoring
    # If the user provided speed column exists, use its Mean or Std
    if feature_col in df.columns:
        # Extract values matrix: (Num_Candidates, Window_Size)
        # Using .values is fast
        values_matrix = df[feature_col].values[candidate_indices]
        # Score = Mean activity in that window (or Std if using coordinates)
        scores = np.nanmean(values_matrix, axis=1) # Handle potential NaNs robustly
    else:
        # Fallback: Use variance of X + Y coordinates if no speed column
        # This detects "did the mouse move geographically?"
        if verbose: print(f"[Info] Feature '{feature_col}' not found. Using X/Y variance.")
        x_cols = [c for c in df.columns if 'x_' in c]
        y_cols = [c for c in df.columns if 'y_' in c]

        if not x_cols or not y_cols:
            raise ValueError(f"Cannot find X/Y coordinate columns. Available columns: {list(df.columns)}")

        x_col = x_cols[0]  # Pick first X
        y_col = y_cols[0]  # Pick first Y

        x_vals = df[x_col].values[candidate_indices]
        y_vals = df[y_col].values[candidate_indices]

        # Score = Std(X) + Std(Y) -> High score means high movement
        scores = np.nanstd(x_vals, axis=1) + np.nanstd(y_vals, axis=1)

    # Handle NaNs in scores (replace with 0)
    scores = np.nan_to_num(scores)

    # 5. APPLY SELECTION STRATEGY
    num_candidates = len(all_starts)
    keep_mask = np.zeros(num_candidates, dtype=bool)

    if strategy == 'quantile':
        # Calculate threshold (e.g., 40th percentile)
        # We keep everything ABOVE this threshold (removing the boring bottom 40%)
        thresh = np.percentile(scores, quantile_threshold * 100)
        keep_mask = scores >= thresh
        if verbose: print(f"  Quantile Strategy: Threshold={thresh:.4f}. Keeping {keep_mask.sum()}/{num_candidates} windows.")

    elif strategy == 'max_variance':
        # Keep top N (let's say top 50% for now, or fixed number)
        # Sort indices by score descending
        sorted_args = np.argsort(-scores)
        n_keep = int(num_candidates * (1 - quantile_threshold))
        keep_idx = sorted_args[:n_keep]
        keep_mask[keep_idx] = True

    elif strategy == 'random_weighted':
        # Probability of selection is proportional to Score
        # Add small epsilon to avoid 0 probability
        probs = scores + 1e-5
        probs = probs / probs.sum()
        # Select N windows based on these weights
        n_keep = min(int(num_candidates * (1 - quantile_threshold)), num_candidates)
        if n_keep > 0:
            selected_idx = np.random.choice(num_candidates, size=n_keep, replace=False, p=probs)
            keep_mask[selected_idx] = True

    else:
        # Default: keep all
        keep_mask[:] = True

    # Filter the starts
    final_starts = all_starts[keep_mask]

    if len(final_starts) == 0:
        if verbose: print("[Warn] Strategy filtered out all windows.")
        return pd.DataFrame()

    # 6. EXTRACT FINAL DATA
    # Re-build indices for only the selected windows
    final_indices = (final_starts[:, None] + window_offsets).ravel()

    result_df = df.iloc[final_indices].copy()

    # 7. METADATA
    num_final = len(final_starts)
    base_id = 900000
    window_ids = np.repeat(np.arange(num_final) + base_id, window_size)
    local_ids = np.tile(np.arange(window_size), num_final)

    result_df.insert(0, 'window_id', window_ids)
    result_df.insert(1, 'local_frame_index', local_ids)

    # Force label
    if isinstance(result_df[action_col].dtype, pd.CategoricalDtype):
         if 'other' not in result_df[action_col].cat.categories:
             result_df[action_col] = result_df[action_col].cat.add_categories('other')
    result_df.loc[:, action_col] = 'other'

    if verbose:
        print(f"Sampled {num_final} background windows (High Variance).")

    return result_df.reset_index(drop=True)


def _sample_background_windows(
    labeled_data: pd.DataFrame,
    window_size: int,
    step_size: int,
    action_col: str = "action",
    verbose: bool = False
) -> pd.DataFrame:
    """
    Extracts 'other' (background) windows from the gaps between actions.

    Logic:
    1. Identifies continuous blocks where action == 'other' (or NaN/None).
    2. Slices these blocks into windows of 'window_size' using 'step_size' (stride).
    3. Assigns a unique window_id (starting from a high number or offset).
    """
    if verbose: print("Sampling background (other) windows...")

    # 1. SETUP
    df = labeled_data.reset_index(drop=True)
    total_frames = len(df)

    # Ensure Unicode strings for fast comparison
    if isinstance(df[action_col].dtype, pd.CategoricalDtype):
        actions = df[action_col].astype(str).to_numpy(dtype='U')
    else:
        # Fill NaNs with 'other' if they exist, to treat them as background
        actions = df[action_col].fillna('other').astype(str).to_numpy(dtype='U')

    # 2. CREATE MASK OF VALID BACKGROUND FRAMES
    # Valid frames are those strictly labeled 'other' (or whatever your background label is)
    # AND potentially 'nan' if you want to use unlabelled data as background
    is_background = (actions == 'other') | (actions == 'nan') | (actions == 'None')

    if not np.any(is_background):
        if verbose: print("[Info] No background frames found.")
        return pd.DataFrame()

    # 3. IDENTIFY CONTINUOUS BLOCKS (Vectorized Run-Length Encoding)
    # We find where the mask changes from True to False or vice versa
    # 0 = False (Action), 1 = True (Background)
    # Diff gives non-zero at boundaries

    # Pad with False to detect start/end of array
    padded_mask = np.concatenate(([False], is_background, [False]))
    diff = np.diff(padded_mask.astype(int))

    # Starts are where diff == 1, Ends are where diff == -1
    bg_starts = np.flatnonzero(diff == 1)
    bg_ends = np.flatnonzero(diff == -1)

    collected_indices = []

    # 4. SLICE BLOCKS WITH STRIDE
    # Iterate through each continuous background block
    for start, end in zip(bg_starts, bg_ends):
        block_len = end - start

        # We need at least one window_size to extract data
        if block_len < window_size:
            continue

        # Generate start indices for this block using the step (stride)
        # range(0, valid_length, step)
        valid_length = block_len - window_size + 1
        local_offsets = np.arange(0, valid_length, step_size)

        # Convert to global indices
        global_start_indices = start + local_offsets

        collected_indices.append(global_start_indices)

    if not collected_indices:
        if verbose: print("[Info] No background blocks long enough for window_size.")
        return pd.DataFrame()

    # 5. BUILD INDICES MATRIX
    all_starts = np.concatenate(collected_indices)
    num_windows = len(all_starts)

    # Broadcasting to create full window indices: (NumWindows, WindowSize)
    window_offsets = np.arange(window_size)
    window_indices = all_starts[:, None] + window_offsets

    # Flatten
    flat_indices = window_indices.ravel()

    # 6. EXTRACT DATA
    result_df = df.iloc[flat_indices].copy()

    # 7. ADD METADATA
    # We use a string prefix or negative ID to distinguish these from action windows
    # Or just continue integer sequence if you manage it externally.
    # Here we use a distinct large range for clarity, e.g., starting at 900000
    base_id = 900000
    window_ids = np.repeat(np.arange(num_windows) + base_id, window_size)
    local_ids = np.tile(np.arange(window_size), num_windows)

    result_df.insert(0, 'window_id', window_ids)
    result_df.insert(1, 'local_frame_index', local_ids)

    # Ensure label is strictly 'other' (in case NaNs were included)
    if isinstance(result_df[action_col].dtype, pd.CategoricalDtype):
        if 'other' not in result_df[action_col].cat.categories:
             result_df[action_col] = result_df[action_col].cat.add_categories('other')

    # Force label to be 'other' (fix any lingering NaNs)
    result_df.loc[:, action_col] = 'other'

    if verbose:
        print(f"Sampled {num_windows} background windows (step={step_size}).")

    return result_df.reset_index(drop=True)


    import pandas as pd
import numpy as np

def merge_training_sets(
    action_data: pd.DataFrame,
    background_data: pd.DataFrame,
    lab_id: str = None,
    video_id: str = None,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Merges action-specific windows with background windows into a single training set.

    Key Features:
    - Safely re-indexes 'window_id' in background_data to ensure uniqueness.
    - Handles pandas Categorical columns (unions categories like 'attack' and 'other').
    - Returns a single, clean DataFrame ready for training.
    """

    # 1. HANDLE EMPTY INPUTS
    if action_data.empty and background_data.empty:
        if verbose: print("[Warn] Both inputs are empty.")
        return pd.DataFrame()

    if action_data.empty:
        if verbose: print("[Info] Action data empty. Returning background only.")
        return background_data.reset_index(drop=True)

    if background_data.empty:
        if verbose: print("[Info] Background data empty. Returning action data only.")
        return action_data.reset_index(drop=True)

    # 2. PREPARE FOR MERGE (Defensive Copies)
    # We copy to avoid SettingWithCopy warnings when modifying window_ids
    df_act = action_data.copy()
    df_bg = background_data.copy()

    # 3. RE-INDEX WINDOW IDs
    # Find the highest window ID in the action set
    max_action_id = df_act['window_id'].max()

    # Shift background IDs so they start immediately after the action IDs
    # e.g., if Action IDs are 0..99, Background IDs will become 100..150
    # We subtract the min first to normalize to 0, then add the offset
    bg_min_id = df_bg['window_id'].min()

    # Safe logic: Normalized ID + (Max Action + 1)
    df_bg['window_id'] = (df_bg['window_id'] - bg_min_id) + (max_action_id + 1)

    # 4. HANDLE CATEGORICAL UNIONS
    # If 'action' is categorical in both, we must ensure they share the same categories
    # before concatenation, otherwise pandas might convert them back to objects.
    if isinstance(df_act['action'].dtype, pd.CategoricalDtype) and isinstance(df_bg['action'].dtype, pd.CategoricalDtype):
        # Union of categories
        combined_cats = set(df_act['action'].cat.categories) | set(df_bg['action'].cat.categories)
        df_act['action'] = df_act['action'].cat.set_categories(combined_cats)
        df_bg['action'] = df_bg['action'].cat.set_categories(combined_cats)

    # 5. CONCATENATE
    # sort=False prevents reordering columns alphabetically
    final_df = pd.concat([df_act, df_bg], ignore_index=True, sort=False)

    # 6. FINAL CLEANUP
    # Ensure window_id is integer (sometimes math operations float it)
    final_df['window_id'] = final_df['window_id'].astype(int)

    # 7.ADDING
    if lab_id is not None:
        final_df.insert(0, 'lab_id', lab_id)
    if video_id is not None:
        final_df.insert(1, 'video_id', video_id)

    if verbose:
        n_act = df_act['window_id'].nunique()
        n_bg = df_bg['window_id'].nunique()
        print(f"Merged Complete.")
        print(f"  - Action Windows:     {n_act}")
        print(f"  - Background Windows: {n_bg}")
        print(f"  - Total Windows:      {final_df['window_id'].nunique()}")
        print(f"  - Total Rows:         {len(final_df)}")

    return final_df


This cell defines the `BehaviorGRU` model, utility functions for training and evaluating it, and advanced techniques for handling imbalanced data and improving rare event detection:
- `BehaviorGRU` class: Defines a Gated Recurrent Unit (GRU) neural network model for behavior classification. It includes a GRU layer and a fully connected output layer, with Xavier uniform initialization for weights.
- `train_gru_model()`: A function to train the `BehaviorGRU` model. It automatically configures model parameters (input_size, output_size, hidden_size) based on the input data, uses `DataLoader` for efficient batch processing, supports weighted `CrossEntropyLoss` for imbalanced classes, and implements early stopping based on validation loss.
- `split_data_stratified()`: Splits data into training and test sets while maintaining the class distribution (stratified sampling). It includes logic to handle rare classes (those with fewer than `min_samples_per_class`) by either placing them entirely in the training set or falling back to a non-stratified split.
- `evaluate_model_per_class()`: Evaluates the trained model and computes detailed per-class metrics (precision, recall, F1-score) along with overall accuracy, a confusion matrix, and a classification report.
- `apply_attack_detection_rules()`: A post-processing function that applies logical rules to model predictions to improve the detection of 'attack' behaviors. It uses a low probability threshold and checks for behavioral context in surrounding frames to refine predictions.
- `predict_with_attack_rules()`: Performs inference with the GRU model and then applies the `apply_attack_detection_rules` function to refine the predictions.
- `evaluate_with_attack_rules()`: Evaluates the model's performance both with and without the attack detection rules, providing a comparison of metrics, especially focusing on improvements in attack-related classes.
- `BehavioralInferenceEngine` class: An advanced inference engine designed to address the "data ceiling" problem for rare behaviors. It uses a low probability threshold, maintains a history of past predictions (context memory), and checks for risky precursor behaviors to refine real-time predictions and reduce false positives for rare events.
- `evaluate_with_behavioral_inference()`: Evaluates the model using the `BehavioralInferenceEngine`, comparing its performance against standard evaluation and highlighting improvements in rare class detection.
- `print_evaluation_results()`: A utility function to format and print the evaluation metrics obtained from the model.
- `compute_class_weights()`: Computes class weights for imbalanced datasets using various strategies (`balanced`, `inverse`, `sqrt`), which can be used in the loss function during training.
- `downsample_majority_classes()`: Intelligently downsamples majority classes based on a specified percentage, helping to reduce class imbalance without removing all samples.
- `smart_balance_dataset()`: Implements a smart balancing strategy that combines downsampling of majority classes with optional oversampling of minority classes (using noise to prevent overfitting).
- `hybrid_balance_dataset()`: A hybrid strategy that oversamples rare classes up to a `target_min_samples` (with added noise) and keeps majority classes as is, then computes new class weights for the resulting distribution.
- `handle_imbalanced_data()`: A comprehensive function to apply various strategies for handling imbalanced class distributions, including weights-only, resampling, hybrid approaches, smart balancing (downsampling + oversampling), complete balancing, and downsampling with weights.

In [ ]:
def aggregate_multi_video_datasets(
    dataset_list: List[Dict[str, Union[pd.DataFrame, str, int]]],
    verbose: bool = True
) -> pd.DataFrame:
    """
    Merges processed datasets from multiple videos/labs into one Master Dataset.

    Parameters
    ----------
    dataset_list : List[Dict]
        A list of dictionaries, where each dict contains:
        {
            'data': pd.DataFrame,   # The output from merge_training_sets
            'lab_id': str/int,      # (Optional) Lab Identifier
            'video_id': str/int     # (Optional) Video Identifier
        }

    Returns
    -------
    pd.DataFrame
        A single DataFrame with globally unique 'window_id's.
        Adds 'lab_id' and 'video_id' columns to every row for tracking.
    """

    if not dataset_list:
        if verbose: print("[Warn] No datasets provided to aggregate.")
        return pd.DataFrame()

    if verbose: print(f"Aggregating {len(dataset_list)} datasets...")

    # 1. INITIALIZE ACCUMULATORS
    accumulated_dfs = []
    global_window_offset = 0
    total_rows = 0

    # We need to collect all possible action categories to unify them later
    all_action_categories = set()

    # 2. ITERATE AND PROCESS
    for entry in dataset_list:
        df = entry['data'].copy() # Defensive copy

        if df.empty:
            continue

        # --- A. STAMP METADATA ---
        # Add lab_id/video_id columns if provided
        if 'lab_id' in entry:
            # Use categorical for memory efficiency if many rows
            df['lab_id'] = entry['lab_id']
            df['lab_id'] = df['lab_id'].astype('category')

        if 'video_id' in entry:
            df['video_id'] = entry['video_id']
            df['video_id'] = df['video_id'].astype('category')

        # --- B. RE-INDEX WINDOW IDS ---
        # Shift this dataset's window_ids so they don't clash with previous ones
        # Normalize local IDs to start at 0 first (safeguard), then add global offset
        local_min = df['window_id'].min()
        df['window_id'] = (df['window_id'] - local_min) + global_window_offset

        # Update offset for the next dataset
        # New Offset = Current Max ID + 1
        current_max = df['window_id'].max()
        global_window_offset = current_max + 1

        # --- C. COLLECT CATEGORIES ---
        if isinstance(df['action'].dtype, pd.CategoricalDtype):
            all_action_categories.update(df['action'].cat.categories)
        else:
            # If strictly strings, just get unique values
            all_action_categories.update(df['action'].unique())

        accumulated_dfs.append(df)
        total_rows += len(df)

    if not accumulated_dfs:
        return pd.DataFrame()

    # 3. UNIFY CATEGORIES BEFORE CONCAT
    # This prevents pandas from reverting Categorical columns back to Object (slow/heavy)
    # We sort categories for consistency
    unified_categories = sorted(list(all_action_categories))

    for i, df in enumerate(accumulated_dfs):
        if 'action' in df.columns:
            # If it's already categorical, we just set the new full list of categories
            if isinstance(df['action'].dtype, pd.CategoricalDtype):
                accumulated_dfs[i]['action'] = df['action'].cat.set_categories(unified_categories)
            else:
                # If it was object/string, convert to categorical now
                accumulated_dfs[i]['action'] = pd.Categorical(df['action'], categories=unified_categories)

    # 4. FINAL CONCATENATION
    master_df = pd.concat(accumulated_dfs, ignore_index=True, sort=False)

    # Ensure window_id is integer
    master_df['window_id'] = master_df['window_id'].astype(int)

    if verbose:
        print("Aggregation Complete.")
        print(f"  - Total Files Merged: {len(accumulated_dfs)}")
        print(f"  - Total Rows:         {len(master_df)}")
        print(f"  - Total Unique Windows: {master_df['window_id'].nunique()}")
        print(f"  - Action Classes:     {len(unified_categories)} {unified_categories}")

    return master_df


@curry
def skip_frames(data: pd.DataFrame, skip_factor: int) -> pd.DataFrame:
    """
    Downsamples the DataFrame by skipping frames.

    Optimizations:
    - Returns immediately if skip_factor is <= 1 (no-op).
    - Resets index to ensure 0..N continuity for downstream loops.
    - Uses .copy() to decouple from original memory (allows GC to free the large parent df).
    """
    if skip_factor <= 1:
        return data

    # 1. Slice with stride
    # We omit ':, ' because iloc[x] implies all columns by default
    downsampled = data.iloc[::skip_factor]

    # 2. Reset Index & Deep Copy
    # reset_index(drop=True) fixes the 0, 2, 4... index problem
    # .copy() ensures we don't hold a reference to the potentially massive original DataFrame
    return downsampled.reset_index(drop=True).copy()


# Feature columns list for model training
FEATURES = [
    'forepaw_left_right_distance',
    'body_length', 'body_orientation', 'face_length', 'face_orientation',
    'nose_forepaw_distance', 'triangle_body_area',
    'face_center_speed_px_per_s', 'face_center_accel_px_per_s2',
    'face_center_speed_cm_per_s', 'face_center_accel_cm_per_s2',
    'body_center_speed_px_per_s', 'body_center_accel_px_per_s2',
    'body_center_speed_cm_per_s', 'body_center_accel_cm_per_s2',
    'tail_base_speed_px_per_s', 'tail_base_accel_px_per_s2',
    'tail_base_speed_cm_per_s', 'tail_base_accel_cm_per_s2',
    'body_turning_rate_deg_per_s', 'body_angular_speed_deg_per_s',
    'face_turning_rate_deg_per_s', 'face_angular_speed_deg_per_s',
    'dist_to_wall_x_px', 'dist_to_wall_y_px', 'dist_to_nearest_wall_cm',
    'dist_to_wall_x_cm', 'dist_to_wall_y_cm', 'dist_to_center_cm'
]


@curry
def normalize_action_labels(df: pd.DataFrame, action_col: str = "action") -> pd.DataFrame:
    """
    Removes 'start_' and 'end_' prefixes from action labels and MERGES the resulting categories.

    Fixes:
    - Handles 'Categorical categories must be unique' error by using mapping instead of renaming.
    - Optimized to work on unique categories (~10 items) rather than rows (~1M items).
    """
    # Defensive copy (optional, removes warnings)
    df = df.copy(deep=False)

    # 1. Ensure Categorical
    if not isinstance(df[action_col].dtype, pd.CategoricalDtype):
        df[action_col] = df[action_col].astype('category')

    # 2. Build Mapping Dictionary
    # Get the unique categories (e.g., ['start_attack', 'end_attack', 'attack'])
    current_cats = df[action_col].cat.categories

    # Calculate new names (e.g., ['attack', 'attack', 'attack'])
    cleaned_cats = current_cats.str.replace(r'^(start_|end_)', '', regex=True)

    # Create a dictionary: {'start_attack': 'attack', 'end_attack': 'attack'}
    # Only map where the name actually changes to save time
    mapping = {old: new for old, new in zip(current_cats, cleaned_cats) if old != new}

    # 3. Apply Mapping
    if mapping:
        # .replace is generally faster/cleaner for merging categories than .map
        # It updates the categories in place or returns a new Series
        df[action_col] = df[action_col].replace(mapping).astype('category')

    return df


@curry
def prepare_data_for_gru(
    df: pd.DataFrame,
    feature_cols: list,
    target_col: str = 'action',
    window_id_col: str = 'window_id',
    frame_id_col: str = 'local_frame_index',
    fill_strategy: str = 'interpolate',
    target_window_size: int = 105
):
    """
    Memory-optimized pipeline to Clean, Normalize, and Reshape data for GRU training.

    Steps:
    1. Sorts data by Window/Frame.
    2. Handles Missing Values.
    3. Pads/Truncates windows to target_window_size.
    4. Normalizes Features (StandardScaler).
    5. Reshapes to (Samples, TimeSteps, Features).
    6. Encodes Labels.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame with features and labels
    feature_cols : list
        List of feature column names
    target_col : str
        Name of target/label column (default: 'action')
    window_id_col : str
        Name of window ID column (default: 'window_id')
    frame_id_col : str
        Name of frame ID column (default: 'local_frame_index')
    fill_strategy : str
        Strategy for handling missing values:
        - 'interpolate' or 'ffill': Fast forward/backward fill (RECOMMENDED, ~1-2 sec)
        - 'window_interpolate': True per-window linear interpolation (slower, ~minutes)
        - 'mean': Fill with column means
        - 'zero': Fill with zeros
    target_window_size : int
        Fixed window size (hidden_size) for the model. All windows will be
        padded (with zeros) or truncated to this size. Default: 105.
        This determines the hidden_size of the GRU model.

    Memory Optimizations:
    - Uses float32 instead of float64 (50% memory reduction)
    - Deletes intermediate variables immediately
    - Processes in-place where possible

    Returns
    -------
    tuple
        (X_tensor, y_tensor, label_encoder, scaler)
    """
    import torch

    # Input validation
    if df.empty:
        raise ValueError("Input DataFrame is empty")
    if not feature_cols:
        raise ValueError("feature_cols cannot be empty")
    if not all(col in df.columns for col in feature_cols):
        missing = [col for col in feature_cols if col not in df.columns]
        raise ValueError(f"Missing feature columns: {missing}")
    if window_id_col not in df.columns:
        raise ValueError(f"Window ID column '{window_id_col}' not found")
    if frame_id_col not in df.columns:
        raise ValueError(f"Frame ID column '{frame_id_col}' not found")
    if target_col not in df.columns:
        raise ValueError(f"Target column '{target_col}' not found")
    if fill_strategy not in ['interpolate', 'mean', 'zero', 'ffill', 'window_interpolate']:
        raise ValueError(f"fill_strategy must be one of ['interpolate', 'ffill', 'mean', 'zero', 'window_interpolate'], got '{fill_strategy}'")

    print("Preparing data for GRU (Memory-Optimized)...")
    print(f"  Input: {len(df):,} rows, {len(feature_cols)} features")
    print(f"  Memory before: {get_memory_usage_mb():.1f} MB")

    # 1. SORT DATA (in-place to save memory)
    print("  - Sorting data...")
    df.sort_values([window_id_col, frame_id_col], inplace=True)
    df.reset_index(drop=True, inplace=True)
    gc.collect()

    # 2. HANDLE MISSING VALUES (NaNs) - OPTIMIZED
    initial_nans = df[feature_cols].isna().sum().sum()
    print(f"  - Handling missing values ({initial_nans:,} NaNs found)...")

    if initial_nans > 0:
        import time
        start_time = time.time()

        if fill_strategy in ['interpolate', 'ffill']:
            # FAST METHOD: Since data is sorted by window_id and frame_id,
            # we can use forward fill + backward fill which is much faster
            # than groupby interpolation and gives similar results for time series
            print("    Using fast forward/backward fill (recommended)...")
            df[feature_cols] = df[feature_cols].ffill().bfill()

        elif fill_strategy == 'window_interpolate':
            # SLOWER BUT MORE ACCURATE: True per-window interpolation
            # Use this only if you have significant gaps within windows
            print("    Using per-window interpolation (slower but more accurate)...")
            print("    Processing columns: ", end="", flush=True)

            # Batch process columns in groups to show progress
            n_cols = len(feature_cols)
            batch_size = max(1, n_cols // 10)

            for i in range(0, n_cols, batch_size):
                batch_cols = feature_cols[i:i+batch_size]
                # Use vectorized interpolation with groupby
                df[batch_cols] = df.groupby(window_id_col, group_keys=False)[batch_cols].apply(
                    lambda g: g.interpolate(method='linear', limit_direction='both').ffill().bfill()
                )
                print(f"{min(i+batch_size, n_cols)}/{n_cols}", end=" ", flush=True)
            print()

        elif fill_strategy == 'mean':
            # Fill with column means
            print("    Using column mean fill...")
            col_means = df[feature_cols].mean()
            df[feature_cols] = df[feature_cols].fillna(col_means)

        elif fill_strategy == 'zero':
            print("    Using zero fill...")
            df[feature_cols] = df[feature_cols].fillna(0)

        # Final fallback: Fill any remaining NaNs with 0
        remaining_nans = df[feature_cols].isna().sum().sum()
        if remaining_nans > 0:
            print(f"    (Filling {remaining_nans:,} remaining NaNs with 0)")
            df[feature_cols] = df[feature_cols].fillna(0)

        elapsed = time.time() - start_time
        print(f"    Completed in {elapsed:.1f}s")
    else:
        print("    No missing values found!")

    gc.collect()

    # 3. ANALYZE WINDOWS AND PAD/TRUNCATE TO FIXED SIZE
    print(f"  - Processing windows (target size: {target_window_size} frames)...")
    num_windows = df[window_id_col].nunique()
    if num_windows == 0:
        raise ValueError("No windows found in the data")

    window_sizes = df.groupby(window_id_col).size()
    if window_sizes.empty:
        raise ValueError("Cannot determine window size - no valid windows")

    # Analyze window size distribution
    min_size = window_sizes.min()
    max_size = window_sizes.max()
    mode_size = int(window_sizes.mode().iloc[0]) if len(window_sizes.mode()) > 0 else int(window_sizes.median())

    print(f"    Window sizes in data: min={min_size}, max={max_size}, mode={mode_size}")
    print(f"    Target window size (hidden_size): {target_window_size}")

    # Get class distribution
    labels_before = df.groupby(window_id_col)[target_col].first()
    class_dist = labels_before.value_counts()
    print(f"    Classes found: {dict(class_dist)}")

    # Use fixed target_window_size - PAD/TRUNCATE ALL windows
    expected_size = target_window_size

    # Count how many windows need padding vs truncation
    need_padding = (window_sizes < expected_size).sum()
    need_truncate = (window_sizes > expected_size).sum()
    exact_match = (window_sizes == expected_size).sum()

    print(f"    Windows: {exact_match} exact, {need_padding} need padding, {need_truncate} need truncation")

    # Process ALL windows with PAD/TRUNCATE
    window_ids = df[window_id_col].unique()
    processed_windows = []
    processed_labels = []

    print(f"    Processing {len(window_ids)} windows...")

    for i, wid in enumerate(window_ids):
        window_data = df[df[window_id_col] == wid][feature_cols].values
        window_label = labels_before[wid]
        current_size = len(window_data)

        if current_size == expected_size:
            # Perfect size - use as is
            processed_windows.append(window_data)
        elif current_size < expected_size:
            # Pad with zeros at the end
            padding = np.zeros((expected_size - current_size, len(feature_cols)), dtype=window_data.dtype)
            processed_windows.append(np.vstack([window_data, padding]))
        else:
            # Truncate (take first expected_size frames)
            processed_windows.append(window_data[:expected_size])

        processed_labels.append(window_label)

        if (i + 1) % 5000 == 0:
            print(f"      Processed {i+1}/{len(window_ids)} windows...")

    # Stack all windows into 3D array
    feature_data = np.stack(processed_windows, axis=0).astype(np.float32)
    num_windows = len(processed_windows)

    # Create label encoder
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(processed_labels)
    print(f"    Labels: {list(label_encoder.classes_)} ({len(label_encoder.classes_)} classes)")

    del processed_windows, processed_labels, window_ids, df
    gc.collect()

    # 4. NORMALIZATION WITH FLOAT32
    print("  - Normalizing features (StandardScaler with float32)...")
    scaler = StandardScaler()

    # Reshape to 2D for scaling (feature_data is already 3D from stack)
    original_shape = feature_data.shape  # (num_windows, expected_size, num_features)
    feature_data_2d = feature_data.reshape(-1, len(feature_cols))

    print(f"    Scaling {len(feature_data_2d):,} samples...")
    scaled_features_2d = scaler.fit_transform(feature_data_2d).astype(np.float32)

    del feature_data, feature_data_2d
    gc.collect()
    print(f"    Memory after scaling: {get_memory_usage_mb():.1f} MB")

    # 5. RESHAPE BACK TO 3D
    print(f"  - Reshaping to ({num_windows}, {expected_size}, {len(feature_cols)})...")
    try:
        X_array = scaled_features_2d.reshape(num_windows, expected_size, len(feature_cols))
        del scaled_features_2d
    except ValueError as e:
        raise ValueError(f"Reshape failed! Error: {e}")

    # 7. CONVERT TO TENSORS (directly from numpy, no extra copy)
    print("  - Converting to PyTorch tensors...")
    X_tensor = torch.from_numpy(X_array)  # Shares memory with numpy array
    y_tensor = torch.tensor(y_encoded, dtype=torch.long)

    # Note: X_tensor shares memory with X_array, so don't delete X_array
    # PyTorch will manage the memory

    print(f"  - Final shape: X={X_tensor.shape}, y={y_tensor.shape}")
    print(f"  - Memory after conversion: {get_memory_usage_mb():.1f} MB")
    print(f"  - Tensor memory: {X_tensor.element_size() * X_tensor.nelement() / 1024**2:.1f} MB")

    return X_tensor, y_tensor, label_encoder, scaler


def subsample_windows(
    df: pd.DataFrame,
    max_windows: int = 100000,
    window_id_col: str = 'window_id',
    target_col: str = 'action',
    stratified: bool = True,
    random_state: int = 42
) -> pd.DataFrame:
    """
    Subsample windows from a large dataset to reduce memory usage.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame with window_id column
    max_windows : int
        Maximum number of windows to keep (default: 100,000)
    window_id_col : str
        Name of window ID column
    target_col : str
        Name of target column for stratified sampling
    stratified : bool
        If True, maintain class distribution when sampling
    random_state : int
        Random seed for reproducibility

    Returns
    -------
    pd.DataFrame
        Subsampled DataFrame
    """
    n_windows = df[window_id_col].nunique()

    if n_windows <= max_windows:
        print(f"  Dataset has {n_windows:,} windows (within limit of {max_windows:,})")
        return df

    print(f"  Subsampling from {n_windows:,} to {max_windows:,} windows...")

    # Get unique window IDs with their labels
    window_labels = df.groupby(window_id_col)[target_col].first().reset_index()

    if stratified and target_col in window_labels.columns:
        # Stratified sampling to maintain class distribution
        from sklearn.model_selection import train_test_split

        # Calculate fraction to keep
        frac = max_windows / n_windows

        try:
            _, sampled_windows = train_test_split(
                window_labels,
                test_size=frac,
                stratify=window_labels[target_col],
                random_state=random_state
            )
            selected_ids = sampled_windows[window_id_col].values
        except ValueError:
            # Fall back to random if stratified fails (e.g., class with only 1 sample)
            print("    Warning: Stratified sampling failed, using random sampling")
            selected_ids = window_labels[window_id_col].sample(
                n=max_windows, random_state=random_state
            ).values
    else:
        # Random sampling
        selected_ids = window_labels[window_id_col].sample(
            n=max_windows, random_state=random_state
        ).values

    # Filter DataFrame
    df_sampled = df[df[window_id_col].isin(selected_ids)].copy()

    print(f"    Kept {len(selected_ids):,} windows ({len(df_sampled):,} rows)")

    # Print class distribution
    if target_col in df_sampled.columns:
        class_counts = df_sampled.groupby(window_id_col)[target_col].first().value_counts()
        print(f"    Class distribution: {dict(class_counts)}")

    return df_sampled


def prepare_data_for_gru_chunked(
    df: pd.DataFrame,
    feature_cols: list,
    target_col: str = 'action',
    window_id_col: str = 'window_id',
    frame_id_col: str = 'local_frame_index',
    max_windows: int = None,
    save_path: str = None,
    target_window_size: int = 105
):
    """
    Memory-efficient version that can subsample and optionally save to disk.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame
    feature_cols : list
        List of feature column names
    target_col : str
        Target column name
    window_id_col : str
        Window ID column name
    frame_id_col : str
        Frame ID column name
    max_windows : int, optional
        Maximum windows to process. If None, processes all.
        Recommended: 100,000-200,000 for 16GB RAM
    save_path : str, optional
        Path to save tensors. If provided, saves and returns paths instead of tensors.
    target_window_size : int
        Fixed window size (hidden_size) for the model. Default: 105.

    Returns
    -------
    tuple
        (X_tensor, y_tensor, label_encoder, scaler) or
        (X_path, y_path, label_encoder, scaler) if save_path provided
    """
    import torch

    print("="*60)
    print("MEMORY-EFFICIENT GRU DATA PREPARATION")
    print("="*60)

    n_windows = df[window_id_col].nunique()
    n_rows = len(df)

    print(f"Input: {n_windows:,} windows, {n_rows:,} rows")
    print(f"Memory: {get_memory_usage_mb():.1f} MB")

    # Subsample if needed
    if max_windows is not None and n_windows > max_windows:
        df = subsample_windows(df, max_windows, window_id_col, target_col)
        gc.collect()

    # Use the optimized prepare function with fixed window size
    X_tensor, y_tensor, label_encoder, scaler = prepare_data_for_gru(
        df, feature_cols, target_col, window_id_col, frame_id_col,
        target_window_size=target_window_size
    )

    # Optionally save to disk
    if save_path is not None:
        os.makedirs(save_path, exist_ok=True)
        X_path = os.path.join(save_path, 'X_tensor.pt')
        y_path = os.path.join(save_path, 'y_tensor.pt')

        print(f"  Saving tensors to disk...")
        torch.save(X_tensor, X_path)
        torch.save(y_tensor, y_path)

        # Clear tensors from memory
        del X_tensor, y_tensor
        gc.collect()

        print(f"    Saved X to: {X_path}")
        print(f"    Saved y to: {y_path}")
        print(f"  Memory after saving: {get_memory_usage_mb():.1f} MB")

        return X_path, y_path, label_encoder, scaler

    return X_tensor, y_tensor, label_encoder, scaler


This cell defines the `main` function, which orchestrates the entire data processing and preparation pipeline for model training. It includes:
- Loading video metadata from `train.csv`.
- Excluding problematic labs (`MABe22_keypoints`, `MABe22_movies`).
- Checking the number of videos and automatically switching to the `main_optimized` function for large datasets (100+ videos) to prevent RAM overflow.
- For smaller datasets, it iterates through video configurations, processes each video using the `DataModeling` class, and collects the resulting datasets.
- Aggregates all processed datasets using `aggregate_multi_video_datasets`.
- Prepares the aggregated data for the GRU model using `prepare_data_for_gru`.
- Applies the specified `balance_strategy` (e.g., 'weights', 'resample', 'hybrid', 'none') to handle class imbalance.
- Includes periodic memory cleanup using `gc.collect()`.

It returns the processed feature tensor (`X_balanced`), label tensor (`y_balanced`), `LabelEncoder`, `StandardScaler`, the master training set, and computed class weights.

In [ ]:
class DataModeling:
    def __init__(self, config: Dict):
        """
        Initializes the DataModeling pipeline with video/lab configuration.
        """
        self.config = config

        # Metadata
        self.lab_id = config.get('lab_id')
        self.video_id = config.get('video_id')
        self.fps = config.get('frames_per_second')
        self.duration = config.get('video_duration_sec')

        # Spatial params
        self.pix_per_cm = config.get('pix_per_cm_approx')
        self.width_pix = config.get('video_width_pix')
        self.height_pix = config.get('video_height_pix')
        self.arena_width_cm = config.get('arena_width_cm')
        self.arena_height_cm = config.get('arena_height_cm')
        self.arena_shape = config.get('arena_shape')
        self.arena_type = config.get('arena_type')

    def process(self, window: int, batch_size: int, skip_n:int) -> pd.DataFrame:
        """
        Orchestrates the full pipeline: Load -> Clean -> Extract Features -> Slice Windows -> Merge.

        Parameters
        ----------
        window : int
            Total size of the training window (frames).
        batch_size : int
            Context size for padding (frames).

        Returns
        -------
        pd.DataFrame
            The final merged training dataset with 'window_id', features, and 'action' label.
        """
        print(f"Processing Lab: {self.lab_id} | Video: {self.video_id}")

        # 1. LOAD DATA
        # (Assuming read_file handles logic for missing files)
        read_anot_data = read_file(
            folder_name='train_annotation',
            lab_id=self.lab_id,
            file_id=self.video_id,
            file_type='parquet'
        )
        read_tracking_data = read_file(
            folder_name='train_tracking',
            lab_id=self.lab_id,
            file_id=self.video_id,
            file_type='parquet'
        )

        if read_anot_data.empty or read_tracking_data.empty:
            print("[Warn] Missing annotation or tracking data. Skipping.")
            return pd.DataFrame()

        # 2. PREPARE ANNOTATIONS
        # Fill gaps and ensure sequence order
        insert_others = insert_other_behaviour_annotations(
            read_anot_data,
            safe_distance=int(self.fps),
            min_frames=int(self.fps)
        )
        # Note: organize_anot_data_action_order likely needs just the dataframe,
        # but if it needs batch_size, use the argument 'batch_size'
        organization = organize_anot_data_action_order(
            insert_others,
            batch_size=batch_size
        )

        # 3. PREPARE TRACKING & FEATURES
        # Interpolate missing frames
        interpolation = interpolation_frames(
            read_tracking_data,
            fps=self.fps,
            t_fps=30
        )

        interpolation = skip_frames(interpolation, skip_factor=skip_n)

        # Extract X/Y coordinates
        extracted_tracking_data, unique_bodyparts = extracting_bodyparts_coordinates(interpolation)

        # Calculate kinematics (velocity, angles, etc.)
        features_df = extracting_features(
            extracted_tracking_data,
            fps=self.fps,
            pix_per_cm=self.pix_per_cm,
            video_width_pix=self.width_pix,
            video_height_pix=self.height_pix,
            unique_bodyparts=unique_bodyparts
        )

        # 4. BUILD LABELED TIMELINE
        # This creates a single DataFrame where every frame is labeled (Start/Action/End/Other)
        # We pass 'batch_size' here if the builder needs to know context size for labeling labels.
        labeled_timeline = build_training_rows_from_annotations_optimized(
            features_data=features_df,
            anot_data=organization,
            window=window,
            fps=self.fps,
            batch_size=batch_size
        )

        # 5. SLICE ACTION WINDOWS
        # Extracts windows based on 'Start -> End' logic
        window_training_rows = slice_windows_iterative_pointer(
            labeled_data=labeled_timeline,
            window_size=window,
            batch_size=batch_size,
            verbose=False
        )

        # 6. SLICE BACKGROUND WINDOWS
        # Extracts 'other' windows from gaps, using 1-second stride (step_size=self.fps)
        other_action_training_rows = sample_background_windows(
            labeled_data=labeled_timeline,
            window_size=window,
            step_size=int(self.fps), # Using int(fps) as step size
            verbose=False
        )

        # 7. MERGE
        final_dataset = merge_training_sets(
            window_training_rows,
            other_action_training_rows,
            lab_id=self.lab_id,
            video_id=self.video_id,
            verbose=True
        )

        final_dataset = normalize_action_labels(final_dataset)


        # Optional: Add metadata columns for global tracking
        if not final_dataset.empty:
            final_dataset['lab_id'] = self.lab_id
            final_dataset['video_id'] = self.video_id

            # Memory optimization: optimize dtypes before returning
            final_dataset = optimize_dataframe_memory(final_dataset, verbose=False)

        # Cleanup intermediate variables to free memory
        del read_anot_data, read_tracking_data
        del insert_others, organization
        del interpolation, extracted_tracking_data, features_df
        del labeled_timeline, window_training_rows, other_action_training_rows
        gc.collect()

        return final_dataset

This cell defines the `train_with_strategy` function and related model configuration utilities. This function implements a complete pipeline for training and evaluating the GRU model with different imbalance handling strategies:
- **Model Configuration**: Automatically infers `input_size`, `output_size`, and `hidden_size` for the `BehaviorGRU` model based on the shape of the input data and the number of unique classes.
- **Stratified Train/Test Split**: Divides the data into training and test sets using stratified sampling to maintain class distribution, even handling rare classes.
- **Imbalance Handling**: Applies the chosen `balance_strategy` (e.g., 'weights', 'resample', 'hybrid', 'smart', 'complete', 'downsample_weights', 'downsample_hybrid', 'none') exclusively to the training data using `handle_imbalanced_data`.
- **Model Training**: Trains the `BehaviorGRU` model using the balanced training data and evaluates its performance on the original, imbalanced test set. It supports early stopping.
- **Evaluation**: Evaluates the trained model using `evaluate_model_per_class` and prints per-class metrics and a classification report.
- **Attack Detection Rules**: Optionally applies post-processing attack detection rules using `evaluate_with_attack_rules` to improve the recall of rare attack behaviors.
- **`create_model_config()`**: A helper function to create a model configuration dictionary, allowing for automatic inference of parameters like `input_size`, `output_size`, and `hidden_size` from provided data.
- **`example_use_behavioral_inference_engine()`**: An example function demonstrating how to use the `BehavioralInferenceEngine` for advanced inference that leverages contextual information and low thresholds to detect rare behaviors, addressing the 'data ceiling' problem.
- **`analyze_training_results()`**: A function to analyze and compare training results from different strategies, providing recommendations based on accuracy, loss, and dataset size.

In [ ]:
class BehaviorGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1, bias=True):
        super(BehaviorGRU, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # GRU Layer
        # batch_first=True means input shape is (batch, seq_len, features)
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False, # Per requirement
            bias=bias            # Per requirement
        )

        # Fully Connected (Output) Layer
        self.fc = nn.Linear(hidden_size, output_size)

        # Apply Xavier (Glorot) Initialization
        self.init_weights()

    def init_weights(self):
        """
        Explicitly applies Xavier Uniform initialization to weights
        to meet the 'Weight Initialization: Xavier' requirement.
        """
        for name, param in self.named_parameters():
            if 'weight' in name:
                init.xavier_uniform_(param)
            elif 'bias' in name:
                init.constant_(param, 0.0)

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)

        # Forward propagate GRU
        # out shape: (batch_size, seq_len, hidden_size)
        # hn shape: (num_layers, batch_size, hidden_size)
        out, _ = self.gru(x)

        # We typically take the output of the *last* time step for classification
        # out[:, -1, :] shape: (batch_size, hidden_size)
        last_time_step = out[:, -1, :]

        # Pass through fully connected layer
        out = self.fc(last_time_step)
        return out


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

def train_gru_model(
    X_tensor: torch.Tensor,
    y_tensor: torch.Tensor,
    config: dict,
    epochs: int = 50,
    learning_rate: float = 0.001,
    class_weights: Optional[torch.Tensor] = None,
    device: Optional[torch.device] = None,
    X_val: Optional[torch.Tensor] = None,
    y_val: Optional[torch.Tensor] = None,
    early_stopping_patience: Optional[int] = None
):
    """
    Trains the GRU model using pre-processed Tensors.

    Parameters
    ----------
    X_tensor : torch.Tensor
        Input features tensor of shape (samples, seq_len, features)
    y_tensor : torch.Tensor
        Target labels tensor of shape (samples,)
    config : dict
        Model configuration dictionary. Values set to None will be auto-configured:
        - input_size: from X_tensor.shape[2]
        - output_size: from unique values in y_tensor
        - hidden_size: from X_tensor.shape[1] (sequence length)
    epochs : int
        Number of training epochs
    learning_rate : float
        Learning rate for optimizer
    class_weights : torch.Tensor, optional
        Class weights for weighted loss. If None, uses uniform weights.
    device : torch.device, optional
        Device to train on. If None, auto-detects CUDA availability.

    Returns
    -------
    model : BehaviorGRU
        Trained model
    """
    # =========================================================================
    # AUTO-CONFIGURE MODEL FROM DATA
    # =========================================================================
    # Infer dimensions from tensor shapes
    n_samples, seq_len, n_features = X_tensor.shape
    n_classes = len(torch.unique(y_tensor))

    # Auto-fill None values in config
    if config.get('input_size') is None:
        config['input_size'] = n_features
        print(f"  [Auto-Config] input_size = {n_features} (from X_tensor features)")

    if config.get('output_size') is None:
        config['output_size'] = n_classes
        print(f"  [Auto-Config] output_size = {n_classes} (from y_tensor classes)")

    if config.get('hidden_size') is None:
        config['hidden_size'] = seq_len
        print(f"  [Auto-Config] hidden_size = {seq_len} (from sequence length)")

    # Set defaults for other config values
    config.setdefault('batch_size', 15)
    config.setdefault('bias', True)

    # Validate config matches data
    if config['input_size'] != n_features:
        print(f"  [Warning] config input_size ({config['input_size']}) != data features ({n_features})")
        print(f"  [Auto-Fix] Updating input_size to {n_features}")
        config['input_size'] = n_features

    if config['output_size'] < n_classes:
        print(f"  [Warning] config output_size ({config['output_size']}) < data classes ({n_classes})")
        print(f"  [Auto-Fix] Updating output_size to {n_classes}")
        config['output_size'] = n_classes

    print(f"\nTraining Model with config:")
    print(f"  - input_size:  {config['input_size']}")
    print(f"  - hidden_size: {config['hidden_size']}")
    print(f"  - output_size: {config['output_size']}")
    print(f"  - batch_size:  {config['batch_size']}")
    print(f"  - bias:        {config.get('bias', True)}")
    print(f"  - Data shape:  {n_samples} samples x {seq_len} timesteps x {n_features} features")

    if class_weights is not None:
        print(f"  - Using class weights: {class_weights.cpu().numpy()}")

    # 1. DATA LOADER
    # Create a TensorDataset and DataLoader directly from the inputs
    dataset = TensorDataset(X_tensor, y_tensor)

    # Shuffle is important for training
    train_loader = DataLoader(dataset, batch_size=config['batch_size'], shuffle=True)

    # 2. INSTANTIATE MODEL
    model = BehaviorGRU(
        input_size=config['input_size'],
        hidden_size=config['hidden_size'],
        output_size=config['output_size'],
        bias=config.get('bias', True)
    )

    # Move to GPU if available
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    print(f"  - Model loaded on {device}")

    # 3. DEFINE LOSS AND OPTIMIZER
    # Use weighted loss if class_weights provided
    if class_weights is not None:
        class_weights = class_weights.to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        print(f"  - Using weighted CrossEntropyLoss")
    else:
        criterion = nn.CrossEntropyLoss()
        print(f"  - Using standard CrossEntropyLoss")

    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Early stopping setup
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    # 4. TRAINING LOOP
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            # Calculate accuracy
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        # Validation
        val_loss = None
        val_acc = None
        if X_val is not None and y_val is not None:
            model.eval()
            val_dataset = TensorDataset(X_val, y_val)
            val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

            val_loss_sum = 0
            val_correct = 0
            val_total = 0

            with torch.no_grad():
                for val_images, val_labels in val_loader:
                    val_images = val_images.to(device)
                    val_labels = val_labels.to(device)
                    val_outputs = model(val_images)
                    val_loss_batch = criterion(val_outputs, val_labels)
                    val_loss_sum += val_loss_batch.item()

                    _, val_predicted = torch.max(val_outputs.data, 1)
                    val_total += val_labels.size(0)
                    val_correct += (val_predicted == val_labels).sum().item()

            val_loss = val_loss_sum / len(val_loader)
            val_acc = 100 * val_correct / val_total

            # Early stopping
            if early_stopping_patience is not None:
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                    best_model_state = model.state_dict().copy()
                else:
                    patience_counter += 1
                    if patience_counter >= early_stopping_patience:
                        print(f"\nEarly stopping at epoch {epoch+1}")
                        model.load_state_dict(best_model_state)
                        break

        # Logging
        if (epoch+1) % 5 == 0:
            avg_loss = total_loss / len(train_loader)
            acc = 100 * correct / total
            if val_loss is not None:
                print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_loss:.4f}, Train Acc: {acc:.2f}%, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
            else:
                print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}, Accuracy: {acc:.2f}%')

    print("Training Complete.")
    return model


def split_data_stratified(
    X_tensor: torch.Tensor,
    y_tensor: torch.Tensor,
    test_size: float = 0.2,
    random_state: int = 42,
    min_samples_per_class: int = 2
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Split data into train and test sets with stratification.
    Handles rare classes (with < min_samples_per_class) by either duplicating them
    or falling back to non-stratified split.

    Parameters
    ----------
    X_tensor : torch.Tensor
        Input features tensor
    y_tensor : torch.Tensor
        Target labels tensor
    test_size : float
        Proportion of data to use for testing
    random_state : int
        Random seed for reproducibility
    min_samples_per_class : int
        Minimum samples required per class for stratification (default: 2)

    Returns
    -------
    X_train, X_test, y_train, y_test : torch.Tensor
        Stratified train/test splits
    """
    from collections import Counter

    # Convert to numpy for sklearn
    X_np = X_tensor.cpu().numpy()
    y_np = y_tensor.cpu().numpy()

    # Check class distribution
    class_counts = Counter(y_np)
    rare_classes = [cls for cls, count in class_counts.items() if count < min_samples_per_class]

    if rare_classes:
        print(f"  ⚠️  Found {len(rare_classes)} rare classes with < {min_samples_per_class} samples")
        print(f"     Rare classes: {rare_classes}")

        # Option 1: Remove rare classes (simplest approach)
        # Option 2: Put all rare class samples in training set, use non-stratified for rest

        # Separate rare and common samples
        rare_mask = np.isin(y_np, rare_classes)
        common_mask = ~rare_mask

        X_rare = X_np[rare_mask]
        y_rare = y_np[rare_mask]
        X_common = X_np[common_mask]
        y_common = y_np[common_mask]

        print(f"     Common samples: {len(X_common)}, Rare samples: {len(X_rare)}")

        if len(X_common) > 0:
            # Stratified split on common classes
            try:
                X_train_common, X_test_common, y_train_common, y_test_common = train_test_split(
                    X_common, y_common,
                    test_size=test_size,
                    random_state=random_state,
                    stratify=y_common
                )
            except ValueError:
                # If still fails, use non-stratified
                print(f"     Falling back to non-stratified split for common classes")
                X_train_common, X_test_common, y_train_common, y_test_common = train_test_split(
                    X_common, y_common,
                    test_size=test_size,
                    random_state=random_state
                )

            # Add ALL rare samples to training set (they need to be learned)
            if len(X_rare) > 0:
                X_train = np.concatenate([X_train_common, X_rare], axis=0)
                y_train = np.concatenate([y_train_common, y_rare], axis=0)
                X_test = X_test_common
                y_test = y_test_common
                print(f"     Added {len(X_rare)} rare samples to training set")
            else:
                X_train, y_train = X_train_common, y_train_common
                X_test, y_test = X_test_common, y_test_common
        else:
            # All samples are rare - just do regular split
            print(f"     All classes are rare - using non-stratified split")
            X_train, X_test, y_train, y_test = train_test_split(
                X_np, y_np,
                test_size=test_size,
                random_state=random_state
            )
    else:
        # Normal stratified split
        X_train, X_test, y_train, y_test = train_test_split(
            X_np, y_np,
            test_size=test_size,
            random_state=random_state,
            stratify=y_np
        )

    # Convert back to tensors
    X_train = torch.tensor(X_train, dtype=torch.float32)
    X_test = torch.tensor(X_test, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    y_test = torch.tensor(y_test, dtype=torch.long)

    return X_train, X_test, y_train, y_test


def evaluate_model_per_class(
    model: torch.nn.Module,
    X_test: torch.Tensor,
    y_test: torch.Tensor,
    label_encoder: LabelEncoder,
    device: Optional[torch.device] = None
) -> Dict[str, Any]:
    """
    Evaluate model and compute per-class metrics.

    Parameters
    ----------
    model : torch.nn.Module
        Trained model
    X_test : torch.Tensor
        Test features
    y_test : torch.Tensor
        Test labels
    label_encoder : LabelEncoder
        Label encoder used during training
    device : torch.device, optional
        Device to run evaluation on

    Returns
    -------
    dict
        Dictionary containing:
        - 'accuracy': Overall accuracy
        - 'per_class_metrics': DataFrame with precision, recall, f1-score per class
        - 'confusion_matrix': Confusion matrix
        - 'classification_report': Text classification report
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model.eval()
    model.to(device)

    # Create test loader
    test_dataset = TensorDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.numpy())

    # Convert to numpy arrays
    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)

    # Compute metrics
    accuracy = (y_true == y_pred).mean() * 100

    # Get all class labels from encoder (to ensure all classes are included)
    all_class_labels = np.arange(len(label_encoder.classes_))
    class_names = label_encoder.classes_

    # Per-class metrics - use labels parameter to ensure all classes are included
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred,
        labels=all_class_labels,  # Include all classes, even if not in test set
        zero_division=0
    )

    # Create DataFrame with per-class metrics
    # Ensure all arrays have the same length
    per_class_df = pd.DataFrame({
        'Class': class_names,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Support': support
    })

    # Confusion matrix - use labels parameter to ensure all classes are included
    cm = confusion_matrix(
        y_true, y_pred,
        labels=all_class_labels
    )

    # Classification report
    report = classification_report(
        y_true, y_pred,
        labels=all_class_labels,
        target_names=class_names,
        zero_division=0
    )

    return {
        'accuracy': accuracy,
        'per_class_metrics': per_class_df,
        'confusion_matrix': cm,
        'classification_report': report
    }


def apply_attack_detection_rules(
    predictions: np.ndarray,
    probabilities: np.ndarray,
    label_encoder: LabelEncoder,
    feature_data: Optional[pd.DataFrame] = None,
    low_threshold: float = 0.3,
    context_window: int = 3,
    attack_class_names: Optional[List[str]] = None
) -> np.ndarray:
    """
    Apply logic rules with low threshold and context check to catch attacks.

    This function post-processes model predictions to improve attack detection by:
    1. Using low threshold to catch borderline attack predictions
    2. Checking context (surrounding frames) for attack patterns
    3. Applying temporal logic rules

    Parameters
    ----------
    predictions : np.ndarray
        Model predictions (class indices) of shape (n_samples,)
    probabilities : np.ndarray
        Model output probabilities of shape (n_samples, n_classes)
    label_encoder : LabelEncoder
        Label encoder to map indices to class names
    feature_data : pd.DataFrame, optional
        Feature data for context checking (must have same length as predictions)
        Should contain features like speed, acceleration, distance metrics
    low_threshold : float
        Low probability threshold for attack detection (default: 0.3)
        If attack probability > low_threshold, consider it a potential attack
    context_window : int
        Number of surrounding frames to check for context (default: 3)
    attack_class_names : list, optional
        List of attack-related class names (default: ['attack', 'chaseattack'])
        If None, auto-detects classes containing 'attack'

    Returns
    -------
    np.ndarray
        Refined predictions after applying logic rules
    """
    if attack_class_names is None:
        # Auto-detect attack classes
        attack_class_names = [name for name in label_encoder.classes_ if 'attack' in name.lower()]

    if not attack_class_names:
        print("Warning: No attack classes found. Returning original predictions.")
        return predictions

    # Get attack class indices
    attack_indices = [np.where(label_encoder.classes_ == name)[0][0]
                     for name in attack_class_names
                     if name in label_encoder.classes_]

    if not attack_indices:
        return predictions

    refined_predictions = predictions.copy()
    n_samples = len(predictions)

    print(f"\n{'='*60}")
    print("Applying Attack Detection Rules (Low Threshold + Context Check)")
    print(f"{'='*60}")
    print(f"Attack classes: {attack_class_names}")
    print(f"Low threshold: {low_threshold}")
    print(f"Context window: {context_window} frames")

    corrections = 0

    # Check each sample
    for i in range(n_samples):
        original_pred = predictions[i]
        probs = probabilities[i]

        # Rule 1: Low Threshold Check
        # If any attack class has probability > low_threshold, flag as potential attack
        max_attack_prob = max([probs[idx] for idx in attack_indices if idx < len(probs)])
        max_attack_idx = attack_indices[np.argmax([probs[idx] for idx in attack_indices if idx < len(probs)])]

        # Rule 2: Context Check
        # Check surrounding frames for attack patterns
        context_start = max(0, i - context_window)
        context_end = min(n_samples, i + context_window + 1)
        context_predictions = predictions[context_start:context_end]
        context_probs = probabilities[context_start:context_end]

        # Count attacks in context
        context_attacks = sum([1 for p in context_predictions if p in attack_indices])
        context_attack_probs = [max([cp[idx] for idx in attack_indices if idx < len(cp)])
                                  for cp in context_probs]
        avg_context_attack_prob = np.mean(context_attack_probs) if context_attack_probs else 0

        # Apply logic rules
        should_be_attack = False

        # Rule 1: Low threshold + high attack probability
        if max_attack_prob > low_threshold:
            # Rule 2: Context check - if surrounding frames also show attacks
            if context_attacks > 0 or avg_context_attack_prob > low_threshold * 0.7:
                should_be_attack = True

        # Rule 3: Strong attack signal in context even if current frame is borderline
        if max_attack_prob > low_threshold * 0.8 and context_attacks >= 2:
            should_be_attack = True

        # Rule 4: If current prediction is not attack but context strongly suggests attack
        if original_pred not in attack_indices:
            if avg_context_attack_prob > low_threshold * 1.2 and context_attacks >= context_window:
                should_be_attack = True

        # Apply correction
        if should_be_attack and original_pred not in attack_indices:
            refined_predictions[i] = max_attack_idx
            corrections += 1

    print(f"\nCorrections made: {corrections}/{n_samples} ({100*corrections/n_samples:.2f}%)")
    print(f"{'='*60}\n")

    return refined_predictions


def predict_with_attack_rules(
    model: torch.nn.Module,
    X_test: torch.Tensor,
    label_encoder: LabelEncoder,
    feature_data: Optional[pd.DataFrame] = None,
    low_threshold: float = 0.3,
    context_window: int = 3,
    attack_class_names: Optional[List[str]] = None,
    device: Optional[torch.device] = None,
    apply_rules: bool = True
) -> Dict[str, Any]:
    """
    Make predictions with attack detection rules applied.

    Parameters
    ----------
    model : torch.nn.Module
        Trained model
    X_test : torch.Tensor
        Test features tensor
    label_encoder : LabelEncoder
        Label encoder
    feature_data : pd.DataFrame, optional
        Feature data for context checking
    low_threshold : float
        Low threshold for attack detection
    context_window : int
        Context window size
    attack_class_names : list, optional
        Attack class names
    device : torch.device, optional
        Device to run on
    apply_rules : bool
        Whether to apply attack detection rules

    Returns
    -------
    dict
        Dictionary with:
        - 'predictions': Final predictions (after rules if apply_rules=True)
        - 'probabilities': Model output probabilities
        - 'raw_predictions': Predictions before applying rules
        - 'corrections': Number of corrections made
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model.eval()
    model.to(device)

    # Create test loader
    test_dataset = TensorDataset(X_test)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    all_probs = []

    with torch.no_grad():
        for X_batch in test_loader:
            X_batch = X_batch[0].to(device)
            outputs = model(X_batch)
            probs = torch.softmax(outputs, dim=1)  # Get probabilities
            all_probs.extend(probs.cpu().numpy())

    probabilities = np.array(all_probs)
    raw_predictions = np.argmax(probabilities, axis=1)

    # Apply attack detection rules
    if apply_rules:
        refined_predictions = apply_attack_detection_rules(
            raw_predictions,
            probabilities,
            label_encoder,
            feature_data,
            low_threshold=low_threshold,
            context_window=context_window,
            attack_class_names=attack_class_names
        )
        corrections = np.sum(raw_predictions != refined_predictions)
    else:
        refined_predictions = raw_predictions
        corrections = 0

    return {
        'predictions': refined_predictions,
        'probabilities': probabilities,
        'raw_predictions': raw_predictions,
        'corrections': corrections
    }


def evaluate_with_attack_rules(
    model: torch.nn.Module,
    X_test: torch.Tensor,
    y_test: torch.Tensor,
    label_encoder: LabelEncoder,
    feature_data: Optional[pd.DataFrame] = None,
    low_threshold: float = 0.3,
    context_window: int = 3,
    attack_class_names: Optional[List[str]] = None,
    device: Optional[torch.device] = None
) -> Dict[str, Any]:
    """
    Evaluate model with attack detection rules and compare with/without rules.

    Parameters
    ----------
    model : torch.nn.Module
        Trained model
    X_test : torch.Tensor
        Test features
    y_test : torch.Tensor
        Test labels
    label_encoder : LabelEncoder
        Label encoder
    feature_data : pd.DataFrame, optional
        Feature data for context checking
    low_threshold : float
        Low threshold for attack detection
    context_window : int
        Context window size
    attack_class_names : list, optional
        Attack class names
    device : torch.device, optional
        Device to run on

    Returns
    -------
    dict
        Evaluation results with and without rules
    """
    print("\n" + "="*80)
    print("EVALUATION WITH ATTACK DETECTION RULES")
    print("="*80)

    # Get predictions with rules
    pred_results = predict_with_attack_rules(
        model, X_test, label_encoder,
        feature_data=feature_data,
        low_threshold=low_threshold,
        context_window=context_window,
        attack_class_names=attack_class_names,
        device=device,
        apply_rules=True
    )

    # Evaluate without rules
    eval_without_rules = evaluate_model_per_class(
        model, X_test, y_test, label_encoder, device=device
    )

    # Evaluate with rules
    y_pred_with_rules = pred_results['predictions']
    y_true = y_test.cpu().numpy()

    # Compute metrics with rules
    all_class_labels = np.arange(len(label_encoder.classes_))
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred_with_rules,
        labels=all_class_labels,
        zero_division=0
    )

    class_names = label_encoder.classes_
    per_class_df = pd.DataFrame({
        'Class': class_names,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Support': support
    })

    accuracy_with_rules = (y_true == y_pred_with_rules).mean() * 100

    # Compare results
    print("\n" + "-"*80)
    print("COMPARISON: Without Rules vs With Rules")
    print("-"*80)
    print(f"Accuracy (without rules): {eval_without_rules['accuracy']:.2f}%")
    print(f"Accuracy (with rules):    {accuracy_with_rules:.2f}%")
    print(f"Improvement:               {accuracy_with_rules - eval_without_rules['accuracy']:.2f}%")
    print(f"Corrections made:          {pred_results['corrections']}/{len(y_true)} ({100*pred_results['corrections']/len(y_true):.2f}%)")

    # Show attack class improvements
    if attack_class_names is None:
        attack_class_names = [name for name in label_encoder.classes_ if 'attack' in name.lower()]

    if attack_class_names:
        print("\n" + "-"*80)
        print("ATTACK CLASS IMPROVEMENTS:")
        print("-"*80)
        for attack_name in attack_class_names:
            if attack_name in label_encoder.classes_:
                attack_idx = np.where(label_encoder.classes_ == attack_name)[0][0]
                without_rules = eval_without_rules['per_class_metrics']
                with_rules = per_class_df

                if attack_idx < len(without_rules) and attack_idx < len(with_rules):
                    recall_without = without_rules.iloc[attack_idx]['Recall']
                    recall_with = with_rules.iloc[attack_idx]['Recall']
                    f1_without = without_rules.iloc[attack_idx]['F1-Score']
                    f1_with = with_rules.iloc[attack_idx]['F1-Score']

                    print(f"\n{attack_name}:")
                    print(f"  Recall: {recall_without:.3f} -> {recall_with:.3f} (Δ{recall_with-recall_without:+.3f})")
                    print(f"  F1-Score: {f1_without:.3f} -> {f1_with:.3f} (Δ{f1_with-f1_without:+.3f})")

    return {
        'without_rules': eval_without_rules,
        'with_rules': {
            'accuracy': accuracy_with_rules,
            'per_class_metrics': per_class_df,
            'corrections': pred_results['corrections']
        },
        'predictions': pred_results
    }


class BehavioralInferenceEngine:
    """
    Advanced inference engine that uses logic rules with low thresholds and context memory
    to catch attacks that the model misses.

    This engine addresses the "data ceiling" problem where models cannot learn to detect
    rare attacks (e.g., 8 training examples) by accepting low-confidence predictions
    (e.g., 10% instead of 90%) and validating them with behavioral context.

    Key Features:
    1. Low Threshold Detection: Catches faint attack signals (>10% probability)
    2. Context Memory: Remembers last N predictions to validate attack patterns
    3. Risky Context Validation: Checks for precursor behaviors (chase, approach, rear)
    4. False Positive Reduction: Rejects attacks without behavioral context
    """

    def __init__(
        self,
        model: torch.nn.Module,
        device: torch.device,
        class_names: np.ndarray,
        low_threshold: float = 0.10,
        history_size: int = 10,
        risky_context: Optional[List[str]] = None
    ):
        """
        Initialize the Behavioral Inference Engine.

        Parameters
        ----------
        model : torch.nn.Module
            Trained model
        device : torch.device
            Device to run inference on
        class_names : np.ndarray
            Array of class names from label encoder
        low_threshold : float
            Low probability threshold for attack detection (default: 0.10)
            If attack probability > low_threshold, flag it
        history_size : int
            Number of previous predictions to remember (default: 10)
        risky_context : list, optional
            List of class names that indicate risky/aggressive context
            (default: ['chase', 'co_rear', 'approach', 'co_chase', 'co_attack'])
        """
        self.model = model
        self.device = device
        self.class_names = class_names
        self.low_threshold = low_threshold
        self.history_size = history_size

        # Create name-to-index mappings
        self.name_to_idx = {name: i for i, name in enumerate(class_names)}
        self.idx_to_name = {i: name for i, name in enumerate(class_names)}

        # Context memory buffer (remembers last N predictions)
        self.history = deque(maxlen=history_size)

        # Risky context behaviors that precede attacks
        if risky_context is None:
            risky_context = ['chase', 'co_rear', 'approach', 'co_chase', 'co_attack', 'rear']
        self.risky_context = risky_context

        # Auto-detect attack classes
        self.attack_classes = [name for name in class_names if 'attack' in name.lower()]
        self.chase_classes = [name for name in class_names if 'chase' in name.lower()]

        print(f"\n{'='*60}")
        print("BehavioralInferenceEngine Initialized")
        print(f"{'='*60}")
        print(f"Low threshold: {low_threshold}")
        print(f"History size: {history_size} frames")
        print(f"Attack classes: {self.attack_classes}")
        print(f"Chase classes: {self.chase_classes}")
        print(f"Risky context: {self.risky_context}")
        print(f"{'='*60}\n")

    def reset_history(self):
        """Reset the context history buffer."""
        self.history.clear()

    def predict_batch(self, X_batch: torch.Tensor) -> np.ndarray:
        """
        Predict on a batch with logic rules applied.

        Processes each sample sequentially to maintain context history.

        Parameters
        ----------
        X_batch : torch.Tensor
            Input batch tensor of shape (batch_size, seq_len, features)

        Returns
        -------
        np.ndarray
            Refined predictions with logic rules applied
        """
        self.model.eval()
        with torch.no_grad():
            X = X_batch.to(self.device)
            logits = self.model(X)
            probs = torch.softmax(logits, dim=1).cpu().numpy()

        final_preds = []

        for i in range(len(probs)):
            p = probs[i]
            pred_idx = np.argmax(p)  # Default decision (highest probability)
            pred_name = self.idx_to_name[pred_idx]

            # --- LOGIC LAYER ---

            # Rule 1: LOW THRESHOLD - Catch faint signals of attack
            # If model is >low_threshold sure it's an attack, flag it
            for attack_name in self.attack_classes:
                if attack_name in self.name_to_idx:
                    atk_id = self.name_to_idx[attack_name]
                    if atk_id < len(p) and p[atk_id] > self.low_threshold:
                        pred_idx = atk_id
                        pred_name = attack_name
                        break

            # Rule 2: LOW THRESHOLD - Catch faint signals of chase
            for chase_name in self.chase_classes:
                if chase_name in self.name_to_idx:
                    chase_id = self.name_to_idx[chase_name]
                    if chase_id < len(p) and p[chase_id] > self.low_threshold:
                        # Only override if not already an attack
                        if pred_name not in self.attack_classes:
                            pred_idx = chase_id
                            pred_name = chase_name
                            break

            # Rule 3: CONTEXT CHECK - Reduce False Positives
            # If we predicted 'attack' or 'chase', did we see risky context recently?
            if pred_name in self.attack_classes + self.chase_classes:
                # Check if we have risky context in history
                has_context = any(
                    h in self.risky_context or h in self.attack_classes or h in self.chase_classes
                    for h in self.history
                )

                # If prediction is Attack/Chase but previous frames were all "Other" (no context),
                # it's likely noise - revert to standard argmax
                if len(self.history) > 5 and not has_context:
                    # Check if there's a strong alternative prediction
                    # (e.g., if 'other' has 0.8 probability, use that instead)
                    p_copy = p.copy()
                    # Suppress the attack/chase prediction
                    for atk_name in self.attack_classes + self.chase_classes:
                        if atk_name in self.name_to_idx:
                            atk_id = self.name_to_idx[atk_name]
                            if atk_id < len(p_copy):
                                p_copy[atk_id] = -1  # Suppress
                    # Use the next best prediction
                    pred_idx = np.argmax(p_copy)
                    pred_name = self.idx_to_name[pred_idx]

            # Update History
            self.history.append(pred_name)
            final_preds.append(pred_idx)

        return np.array(final_preds)

    def predict_sequential(
        self,
        X_test: torch.Tensor,
        batch_size: int = 1,
        shuffle: bool = False
    ) -> np.ndarray:
        """
        Predict on entire dataset sequentially (maintains context across batches).

        Parameters
        ----------
        X_test : torch.Tensor
            Test features tensor
        batch_size : int
            Batch size for processing (default: 1 for sequential context)
        shuffle : bool
            Whether to shuffle (should be False to maintain temporal context)

        Returns
        -------
        np.ndarray
            All predictions with logic rules applied
        """
        self.reset_history()

        test_dataset = TensorDataset(X_test)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=shuffle)

        all_preds = []

        for X_batch in test_loader:
            X_b = X_batch[0]
            preds = self.predict_batch(X_b)
            all_preds.extend(preds)

        return np.array(all_preds)


def evaluate_with_behavioral_inference(
    model: torch.nn.Module,
    X_test: torch.Tensor,
    y_test: torch.Tensor,
    label_encoder: LabelEncoder,
    low_threshold: float = 0.10,
    history_size: int = 10,
    risky_context: Optional[List[str]] = None,
    device: Optional[torch.device] = None
) -> Dict[str, Any]:
    """
    Evaluate model using BehavioralInferenceEngine and compare with standard evaluation.

    This is the "Magic Fix" that addresses the data ceiling problem by accepting
    low-confidence attack predictions and validating them with behavioral context.

    Parameters
    ----------
    model : torch.nn.Module
        Trained model
    X_test : torch.Tensor
        Test features
    y_test : torch.Tensor
        Test labels
    label_encoder : LabelEncoder
        Label encoder
    low_threshold : float
        Low threshold for attack detection (default: 0.10)
    history_size : int
        Context history size (default: 10)
    risky_context : list, optional
        Risky context behaviors
    device : torch.device, optional
        Device to run on

    Returns
    -------
    dict
        Evaluation results with and without behavioral inference
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print("\n" + "="*80)
    print("EVALUATION WITH BEHAVIORAL INFERENCE ENGINE")
    print("="*80)
    print("(The 'Magic Fix' for catching attacks with low confidence)")
    print("="*80)

    # Standard evaluation (without logic rules)
    print("\nStep 1: Standard Model Evaluation (Baseline)")
    print("-"*80)
    eval_standard = evaluate_model_per_class(
        model, X_test, y_test, label_encoder, device=device
    )

    # Behavioral inference evaluation (with logic rules)
    print("\nStep 2: Behavioral Inference Engine (Logic Rules)")
    print("-"*80)
    engine = BehavioralInferenceEngine(
        model, device, label_encoder.classes_,
        low_threshold=low_threshold,
        history_size=history_size,
        risky_context=risky_context
    )

    # Get predictions with behavioral inference
    y_pred_inference = engine.predict_sequential(X_test, batch_size=1, shuffle=False)
    y_true = y_test.cpu().numpy()

    # Compute metrics with inference
    all_class_labels = np.arange(len(label_encoder.classes_))
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred_inference,
        labels=all_class_labels,
        zero_division=0
    )

    class_names = label_encoder.classes_
    per_class_df = pd.DataFrame({
        'Class': class_names,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Support': support
    })

    accuracy_inference = (y_true == y_pred_inference).mean() * 100

    # Confusion matrix
    cm = confusion_matrix(
        y_true, y_pred_inference,
        labels=all_class_labels
    )

    # Classification report
    report = classification_report(
        y_true, y_pred_inference,
        labels=all_class_labels,
        target_names=class_names,
        zero_division=0
    )

    # Compare results
    print("\n" + "="*80)
    print("COMPARISON: Standard vs Behavioral Inference")
    print("="*80)
    print(f"Accuracy (standard):        {eval_standard['accuracy']:.2f}%")
    print(f"Accuracy (with inference):  {accuracy_inference:.2f}%")
    print(f"Improvement:                 {accuracy_inference - eval_standard['accuracy']:+.2f}%")

    # Show attack class improvements
    attack_classes = [name for name in label_encoder.classes_ if 'attack' in name.lower()]
    chase_classes = [name for name in label_encoder.classes_ if 'chase' in name.lower()]
    rare_classes = attack_classes + chase_classes

    if rare_classes:
        print("\n" + "-"*80)
        print("RARE CLASS IMPROVEMENTS (Attack/Chase):")
        print("-"*80)
        for class_name in rare_classes:
            if class_name in label_encoder.classes_:
                class_idx = np.where(label_encoder.classes_ == class_name)[0][0]
                standard = eval_standard['per_class_metrics']
                inference = per_class_df

                if class_idx < len(standard) and class_idx < len(inference):
                    recall_std = standard.iloc[class_idx]['Recall']
                    recall_inf = inference.iloc[class_idx]['Recall']
                    f1_std = standard.iloc[class_idx]['F1-Score']
                    f1_inf = inference.iloc[class_idx]['F1-Score']

                    print(f"\n{class_name}:")
                    print(f"  Recall:   {recall_std:.3f} -> {recall_inf:.3f} (Δ{recall_inf-recall_std:+.3f})")
                    print(f"  F1-Score: {f1_std:.3f} -> {f1_inf:.3f} (Δ{f1_inf-f1_std:+.3f})")

    print("\n" + "-"*80)
    print("CLASSIFICATION REPORT (With Behavioral Inference):")
    print("-"*80)
    print(report)

    return {
        'standard': eval_standard,
        'inference': {
            'accuracy': accuracy_inference,
            'per_class_metrics': per_class_df,
            'confusion_matrix': cm,
            'classification_report': report
        },
        'predictions': y_pred_inference,
        'engine': engine
    }


def print_evaluation_results(eval_results: Dict[str, Any], strategy_name: str = "") -> None:
    """
    Print evaluation results in a formatted way.

    Parameters
    ----------
    eval_results : dict
        Results from evaluate_model_per_class
    strategy_name : str
        Name of the strategy for display
    """
    print("\n" + "="*80)
    if strategy_name:
        print(f"EVALUATION RESULTS - {strategy_name.upper()} STRATEGY")
    else:
        print("EVALUATION RESULTS")
    print("="*80)

    print(f"\nOverall Accuracy: {eval_results['accuracy']:.2f}%")

    print("\n" + "-"*80)
    print("PER-CLASS METRICS:")
    print("-"*80)
    print(eval_results['per_class_metrics'].to_string(index=False))

    print("\n" + "-"*80)
    print("CLASSIFICATION REPORT:")
    print("-"*80)
    print(eval_results['classification_report'])

    # Summary statistics
    df = eval_results['per_class_metrics']
    print("\n" + "-"*80)
    print("SUMMARY STATISTICS:")
    print("-"*80)
    print(f"Mean Precision: {df['Precision'].mean():.3f}")
    print(f"Mean Recall: {df['Recall'].mean():.3f}")
    print(f"Mean F1-Score: {df['F1-Score'].mean():.3f}")
    print(f"Weighted Avg F1: {(df['F1-Score'] * df['Support']).sum() / df['Support'].sum():.3f}")


import torch
import numpy as np
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight


def compute_class_weights(y_tensor: torch.Tensor, strategy: str = 'balanced') -> torch.Tensor:
    """
    Compute class weights from label tensor.

    Parameters
    ----------
    y_tensor : torch.Tensor
        Label tensor of shape (samples,)
    strategy : str
        Weight computation strategy:
        - 'balanced': sklearn's balanced weights (n_samples / (n_classes * np.bincount(y)))
        - 'inverse': Inverse frequency weights
        - 'sqrt': Square root of inverse frequency

    Returns
    -------
    torch.Tensor
        Class weights tensor of shape (n_classes,)
    """
    y_np = y_tensor.cpu().numpy()
    classes_unique = np.unique(y_np)
    n_classes = len(classes_unique)

    if strategy == 'balanced':
        weights_array = compute_class_weight(
            class_weight='balanced',
            classes=classes_unique,
            y=y_np
        )
    elif strategy == 'inverse':
        counts = Counter(y_np)
        total = len(y_np)
        weights_array = np.array([total / (n_classes * counts[c]) for c in classes_unique])
    elif strategy == 'sqrt':
        counts = Counter(y_np)
        total = len(y_np)
        weights_array = np.array([np.sqrt(total / (n_classes * counts[c])) for c in classes_unique])
    else:
        raise ValueError(f"Unknown strategy: {strategy}. Choose from ['balanced', 'inverse', 'sqrt']")

    # Normalize weights to have mean=1 (optional, helps with learning rate stability)
    weights_array = weights_array / weights_array.mean()

    return torch.tensor(weights_array, dtype=torch.float32)


def downsample_majority_classes(
    X_tensor: torch.Tensor,
    y_tensor: torch.Tensor,
    label_encoder: LabelEncoder,
    downsample_config: dict
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Intelligently downsample majority classes by percentage.

    Parameters
    ----------
    X_tensor : torch.Tensor
        Input features tensor
    y_tensor : torch.Tensor
        Target labels tensor
    label_encoder : LabelEncoder
        Label encoder to map class names to indices
    downsample_config : dict
        Configuration for downsampling:
        - 'classes_to_downsample': list of class names (e.g., ['other', 'co_other'])
        - 'downsample_percentage': float (0.0-1.0), percentage to keep (e.g., 0.3 = keep 30%)
        - 'method': str, 'random' or 'stratified' (default: 'random')
        - 'random_state': int, random seed (default: 42)

    Returns
    -------
    X_downsampled : torch.Tensor
        Downsampled features tensor
    y_downsampled : torch.Tensor
        Downsampled labels tensor
    """
    X_np = X_tensor.cpu().numpy()
    y_np = y_tensor.cpu().numpy()

    # Get class names to downsample
    classes_to_downsample = downsample_config.get('classes_to_downsample', [])
    downsample_pct = downsample_config.get('downsample_percentage', 0.3)
    method = downsample_config.get('method', 'random')
    random_state = downsample_config.get('random_state', 42)

    if not classes_to_downsample:
        print("  No classes specified for downsampling. Returning original data.")
        return X_tensor, y_tensor

    # Map class names to indices
    class_name_to_idx = {name: idx for idx, name in enumerate(label_encoder.classes_)}
    classes_to_downsample_idx = []
    for class_name in classes_to_downsample:
        if class_name in class_name_to_idx:
            classes_to_downsample_idx.append(class_name_to_idx[class_name])
        else:
            print(f"  Warning: Class '{class_name}' not found in label encoder. Skipping.")

    if not classes_to_downsample_idx:
        print("  No valid classes found for downsampling. Returning original data.")
        return X_tensor, y_tensor

    print(f"\nDownsampling majority classes: {classes_to_downsample}")
    print(f"  Downsample percentage: {downsample_pct*100:.1f}% (keeping {downsample_pct*100:.1f}% of samples)")

    # Separate data into classes to downsample and classes to keep
    mask_to_downsample = np.isin(y_np, classes_to_downsample_idx)
    indices_to_downsample = np.where(mask_to_downsample)[0]
    indices_to_keep = np.where(~mask_to_downsample)[0]

    # Downsample the specified classes
    if len(indices_to_downsample) > 0:
        n_keep = int(len(indices_to_downsample) * downsample_pct)

        if method == 'random':
            np.random.seed(random_state)
            selected_indices = np.random.choice(
                indices_to_downsample,
                size=n_keep,
                replace=False
            )
        elif method == 'stratified':
            # Stratified sampling: maintain distribution within the class
            # For now, use random (can be enhanced to maintain temporal distribution)
            np.random.seed(random_state)
            selected_indices = np.random.choice(
                indices_to_downsample,
                size=n_keep,
                replace=False
            )
        else:
            raise ValueError(f"Unknown method: {method}. Use 'random' or 'stratified'")

        print(f"  - Downsampled {len(indices_to_downsample)} -> {n_keep} samples")

        # Combine downsampled indices with indices to keep
        final_indices = np.concatenate([indices_to_keep, selected_indices])
    else:
        final_indices = indices_to_keep
        print(f"  - No samples found for specified classes. Keeping all data.")

    # Sort indices to maintain order
    final_indices = np.sort(final_indices)

    # Create downsampled arrays
    X_downsampled = X_np[final_indices]
    y_downsampled = y_np[final_indices]

    print(f"  - Final dataset size: {len(X_downsampled):,} (Original: {len(X_tensor):,})")

    return (
        torch.tensor(X_downsampled, dtype=torch.float32),
        torch.tensor(y_downsampled, dtype=torch.long)
    )


def smart_balance_dataset(
    X_tensor: torch.Tensor,
    y_tensor: torch.Tensor,
    label_encoder: LabelEncoder,
    downsample_config: dict,
    oversample_config: Optional[dict] = None
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Smart balancing: Downsample majority classes + optionally oversample minority classes.

    Parameters
    ----------
    X_tensor : torch.Tensor
        Input features tensor
    y_tensor : torch.Tensor
        Target labels tensor
    label_encoder : LabelEncoder
        Label encoder to map class names to indices
    downsample_config : dict
        Configuration for downsampling majority classes:
        - 'classes_to_downsample': list of class names (e.g., ['other', 'co_other'])
        - 'downsample_percentage': float (0.0-1.0), percentage to keep
        - 'method': str, 'random' or 'stratified' (default: 'random')
        - 'random_state': int, random seed (default: 42)
    oversample_config : dict, optional
        Configuration for oversampling minority classes:
        - 'target_min_samples': int, minimum samples per class after oversampling
        - 'noise_scale': float, noise to add during oversampling (default: 0.005)
        If None, only downsampling is performed.

    Returns
    -------
    X_balanced : torch.Tensor
        Balanced features tensor
    y_balanced : torch.Tensor
        Balanced labels tensor
    class_weights : torch.Tensor
        Class weights for the balanced distribution
    """
    print(f"\n{'='*60}")
    print("Smart Balancing: Downsampling + Oversampling")
    print(f"{'='*60}")

    # Step 1: Downsample majority classes
    X_downsampled, y_downsampled = downsample_majority_classes(
        X_tensor, y_tensor, label_encoder, downsample_config
    )

    # Step 2: Optionally oversample minority classes
    if oversample_config is not None:
        target_min_samples = oversample_config.get('target_min_samples', 1000)
        noise_scale = oversample_config.get('noise_scale', 0.005)

        print(f"\nOversampling minority classes to {target_min_samples} samples...")
        X_balanced, y_balanced, _ = hybrid_balance_dataset(
            X_downsampled, y_downsampled,
            target_min_samples=target_min_samples,
            noise_scale=noise_scale
        )
    else:
        X_balanced = X_downsampled
        y_balanced = y_downsampled

    # Step 3: Compute class weights for final distribution
    classes_unique = np.unique(y_balanced.cpu().numpy())
    weights_array = compute_class_weight(
        class_weight='balanced',
        classes=classes_unique,
        y=y_balanced.cpu().numpy()
    )

    # Print final distribution
    y_balanced_np = y_balanced.cpu().numpy()
    counts_after = Counter(y_balanced_np)
    print("\nFinal Class Distribution:")
    for class_idx, count in sorted(counts_after.items()):
        try:
            class_name = label_encoder.classes_[class_idx] if class_idx < len(label_encoder.classes_) else f"Class_{class_idx}"
        except (IndexError, TypeError):
            class_name = f"Class_{class_idx}"
        print(f"  {class_name}: {count:6d} samples ({100*count/len(y_balanced_np):.2f}%)")

    return (
        X_balanced,
        y_balanced,
        torch.tensor(weights_array, dtype=torch.float32)
    )


def hybrid_balance_dataset(X_train, y_train, target_min_samples=5000, noise_scale=0.005):
    """
    Hybrid Strategy:
    1. OVERSAMPLE rare classes up to 'target_min_samples' (with noise).
    2. LEAVE majority classes alone (to preserve real distribution).
    3. COMPUTE weights for the new distribution to handle remaining imbalance.

    Parameters
    ----------
    X_train : torch.Tensor
        Input features tensor
    y_train : torch.Tensor
        Target labels tensor
    target_min_samples : int
        Minimum number of samples per class after oversampling
    noise_scale : float
        Standard deviation of noise to add during oversampling

    Returns:
        X_hybrid, y_hybrid: The balanced tensors.
        new_weights: Tensor of class weights for the loss function.
    """
    print(f"Applying Hybrid Balancing (Target Min Samples: {target_min_samples})...")

    X_np = X_train.cpu().numpy()
    y_np = y_train.cpu().numpy()

    counts = Counter(y_np)
    X_list = []
    y_list = []

    # 1. Resampling Loop
    for class_idx, count in counts.items():
        indices = np.where(y_np == class_idx)[0]
        X_class = X_np[indices]
        y_class = y_np[indices]

        if count < target_min_samples:
            # --- OVERSAMPLE ---
            # We need to add (target - count) samples
            diff = target_min_samples - count
            print(f"  - Class {class_idx}: Boosting {count} -> {target_min_samples} samples")

            # Random choice with replacement
            random_indices = np.random.choice(indices, size=diff, replace=True)
            X_extra = X_np[random_indices]
            y_extra = y_np[random_indices]

            # Add Noise (Jitter) to prevent exact memorization
            noise = np.random.normal(0, noise_scale, X_extra.shape)
            X_extra = X_extra + noise

            # Combine
            X_class = np.concatenate([X_class, X_extra], axis=0)
            y_class = np.concatenate([y_class, y_extra], axis=0)

        else:
            # --- KEEP AS IS ---
            # This class is already abundant (e.g., 'other' with 200k)
            pass

        X_list.append(X_class)
        y_list.append(y_class)

    # Reassemble
    X_final = np.concatenate(X_list, axis=0)
    y_final = np.concatenate(y_list, axis=0)

    print(f"  - New Dataset Size: {len(X_final)} (Original: {len(X_train)})")

    # 2. Compute New Weights (for the Remaining Imbalance)
    classes_unique = np.unique(y_final)
    weights_array = compute_class_weight(
        class_weight='balanced',
        classes=classes_unique,
        y=y_final
    )

    return (
        torch.tensor(X_final, dtype=torch.float32),
        torch.tensor(y_final, dtype=torch.long),
        torch.tensor(weights_array, dtype=torch.float32)
    )


def handle_imbalanced_data(
    X_tensor: torch.Tensor,
    y_tensor: torch.Tensor,
    strategy: str = 'weights',
    resample_config: Optional[dict] = None,
    weight_config: Optional[dict] = None,
    label_encoder: Optional[LabelEncoder] = None,
    downsample_config: Optional[dict] = None
) -> Tuple[torch.Tensor, torch.Tensor, Optional[torch.Tensor]]:
    """
    Handle imbalanced class distribution using specified strategy.

    Parameters
    ----------
    X_tensor : torch.Tensor
        Input features tensor
    y_tensor : torch.Tensor
        Target labels tensor
    strategy : str
        Strategy to use:
        - 'weights': Use class weights in loss function (no resampling)
        - 'resample': Resample data to balance classes
        - 'hybrid': Combine resampling + weights
        - 'smart': Intelligently downsample majority classes + oversample minority classes
        - 'complete': Complete strategy (Downsample + Resample + Weighting) - Most thorough
        - 'downsample_weights': Downsample majority classes + use class weights (no oversampling)
        - 'downsample_hybrid': Downsample majority classes + Hybrid (Resample + Weights)
        - 'none': No balancing (use uniform weights)
    resample_config : dict, optional
        Configuration for resampling:
        - target_min_samples: int (default: 1000)
        - noise_scale: float (default: 0.005)
    weight_config : dict, optional
        Configuration for weight computation:
        - weight_strategy: str (default: 'balanced')
          Options: 'balanced', 'inverse', 'sqrt'
    label_encoder : LabelEncoder, optional
        Label encoder for mapping class names to indices (required for 'smart' strategy)
    downsample_config : dict, optional
        Configuration for downsampling (required for 'smart' strategy):
        - 'classes_to_downsample': list of class names (e.g., ['other', 'co_other'])
        - 'downsample_percentage': float (0.0-1.0), percentage to keep (e.g., 0.3 = keep 30%)
        - 'method': str, 'random' or 'stratified' (default: 'random')
        - 'random_state': int, random seed (default: 42)

    Returns
    -------
    X_balanced : torch.Tensor
        Balanced/resampled features tensor
    y_balanced : torch.Tensor
        Balanced/resampled labels tensor
    class_weights : torch.Tensor or None
        Class weights tensor if strategy uses weights, None otherwise
    """
    if resample_config is None:
        resample_config = {'target_min_samples': 1000, 'noise_scale': 0.005}  # Lower default for production
    if weight_config is None:
        weight_config = {'weight_strategy': 'balanced'}

    print(f"\n{'='*60}")
    print(f"Handling Imbalanced Data - Strategy: {strategy}")
    print(f"{'='*60}")

    # Print class distribution before balancing
    y_np = y_tensor.cpu().numpy()
    counts_before = Counter(y_np)
    print("\nClass Distribution (Before):")
    for class_idx, count in sorted(counts_before.items()):
        print(f"  Class {class_idx}: {count:6d} samples ({100*count/len(y_np):.2f}%)")

    if strategy == 'weights':
        # Only compute weights, no resampling
        print("\nStrategy: Using class weights in loss function (no resampling)")
        class_weights = compute_class_weights(y_tensor, strategy=weight_config['weight_strategy'])
        print(f"Computed class weights: {class_weights.cpu().numpy()}")
        return X_tensor, y_tensor, class_weights

    elif strategy == 'resample':
        # Resample only, compute weights for remaining imbalance
        print("\nStrategy: Resampling data to balance classes")
        X_resampled, y_resampled, class_weights = hybrid_balance_dataset(
            X_tensor, y_tensor,
            target_min_samples=resample_config['target_min_samples'],
            noise_scale=resample_config['noise_scale']
        )

        # Print class distribution after resampling
        y_resampled_np = y_resampled.cpu().numpy()
        counts_after = Counter(y_resampled_np)
        print("\nClass Distribution (After Resampling):")
        for class_idx, count in sorted(counts_after.items()):
            print(f"  Class {class_idx}: {count:6d} samples ({100*count/len(y_resampled_np):.2f}%)")

        return X_resampled, y_resampled, class_weights

    elif strategy == 'hybrid':
        # Resample + weights
        print("\nStrategy: Hybrid (Resampling + Class Weights)")
        X_resampled, y_resampled, _ = hybrid_balance_dataset(
            X_tensor, y_tensor,
            target_min_samples=resample_config['target_min_samples'],
            noise_scale=resample_config['noise_scale']
        )
        # Compute additional weights for remaining imbalance
        class_weights = compute_class_weights(y_resampled, strategy=weight_config['weight_strategy'])

        # Print class distribution after resampling
        y_resampled_np = y_resampled.cpu().numpy()
        counts_after = Counter(y_resampled_np)
        print("\nClass Distribution (After Resampling):")
        for class_idx, count in sorted(counts_after.items()):
            print(f"  Class {class_idx}: {count:6d} samples ({100*count/len(y_resampled_np):.2f}%)")
        print(f"Additional class weights: {class_weights.cpu().numpy()}")

        return X_resampled, y_resampled, class_weights

    elif strategy == 'smart':
        # Smart balancing: Downsample majority + oversample minority
        if label_encoder is None:
            raise ValueError("label_encoder is required for 'smart' strategy")
        if downsample_config is None:
            raise ValueError("downsample_config is required for 'smart' strategy")

        print("\nStrategy: Smart Balancing (Downsample Majority + Oversample Minority)")

        # Use oversample_config from resample_config if provided
        oversample_config = None
        if resample_config:
            oversample_config = {
                'target_min_samples': resample_config.get('target_min_samples', 1000),
                'noise_scale': resample_config.get('noise_scale', 0.005)
            }

        X_balanced, y_balanced, class_weights = smart_balance_dataset(
            X_tensor, y_tensor, label_encoder,
            downsample_config=downsample_config,
            oversample_config=oversample_config
        )

        return X_balanced, y_balanced, class_weights

    elif strategy == 'complete':
        # Complete strategy: Downsample + Resample + Weighting
        if label_encoder is None:
            raise ValueError("label_encoder is required for 'complete' strategy")
        if downsample_config is None:
            raise ValueError("downsample_config is required for 'complete' strategy")

        print("\nStrategy: Complete Balancing (Downsample + Resample + Weighting)")
        print("="*60)

        # Step 1: Downsample majority classes
        print("\nStep 1: Downsampling majority classes...")
        X_downsampled, y_downsampled = downsample_majority_classes(
            X_tensor, y_tensor, label_encoder, downsample_config
        )

        # Step 2: Oversample minority classes
        if resample_config is None:
            resample_config = {'target_min_samples': 1000, 'noise_scale': 0.005}

        print(f"\nStep 2: Oversampling minority classes to {resample_config['target_min_samples']} samples...")
        X_resampled, y_resampled, _ = hybrid_balance_dataset(
            X_downsampled, y_downsampled,
            target_min_samples=resample_config['target_min_samples'],
            noise_scale=resample_config['noise_scale']
        )

        # Step 3: Compute additional class weights for fine-tuning
        print(f"\nStep 3: Computing class weights for remaining imbalance...")
        class_weights = compute_class_weights(
            y_resampled,
            strategy=weight_config.get('weight_strategy', 'balanced')
        )

        # Print final distribution
        y_resampled_np = y_resampled.cpu().numpy()
        counts_after = Counter(y_resampled_np)
        print("\nFinal Class Distribution (After Complete Balancing):")
        for class_idx, count in sorted(counts_after.items()):
            try:
                class_name = label_encoder.classes_[class_idx] if class_idx < len(label_encoder.classes_) else f"Class_{class_idx}"
            except (IndexError, TypeError):
                class_name = f"Class_{class_idx}"
            print(f"  {class_name}: {count:6d} samples ({100*count/len(y_resampled_np):.2f}%)")
        print(f"\nFinal class weights: {class_weights.cpu().numpy()}")

        return X_resampled, y_resampled, class_weights

    elif strategy == 'downsample_weights':
        # Downsample majority classes + use class weights (no oversampling)
        if label_encoder is None:
            raise ValueError("label_encoder is required for 'downsample_weights' strategy")
        if downsample_config is None:
            raise ValueError("downsample_config is required for 'downsample_weights' strategy")

        print("\nStrategy: Downsample Majority + Class Weights (No Oversampling)")
        print("="*60)

        # Step 1: Downsample majority classes
        print("\nStep 1: Downsampling majority classes...")
        X_downsampled, y_downsampled = downsample_majority_classes(
            X_tensor, y_tensor, label_encoder, downsample_config
        )

        # Step 2: Compute class weights for the downsampled distribution
        print(f"\nStep 2: Computing class weights for downsampled distribution...")
        class_weights = compute_class_weights(
            y_downsampled,
            strategy=weight_config.get('weight_strategy', 'balanced')
        )

        # Print final distribution
        y_downsampled_np = y_downsampled.cpu().numpy()
        counts_after = Counter(y_downsampled_np)
        print("\nFinal Class Distribution (After Downsampling):")
        for class_idx, count in sorted(counts_after.items()):
            try:
                class_name = label_encoder.classes_[class_idx] if class_idx < len(label_encoder.classes_) else f"Class_{class_idx}"
            except (IndexError, TypeError):
                class_name = f"Class_{class_idx}"
            print(f"  {class_name}: {count:6d} samples ({100*count/len(y_downsampled_np):.2f}%)")
        print(f"\nComputed class weights: {class_weights.cpu().numpy()}")

        return X_downsampled, y_downsampled, class_weights

    elif strategy == 'downsample_hybrid':
        # Downsample majority classes + Hybrid (Resample + Weights)
        if label_encoder is None:
            raise ValueError("label_encoder is required for 'downsample_hybrid' strategy")
        if downsample_config is None:
            raise ValueError("downsample_config is required for 'downsample_hybrid' strategy")

        print("\nStrategy: Downsample Majority + Hybrid (Resample + Weights)")
        print("="*60)

        # Step 1: Downsample majority classes
        print("\nStep 1: Downsampling majority classes...")
        X_downsampled, y_downsampled = downsample_majority_classes(
            X_tensor, y_tensor, label_encoder, downsample_config
        )

        # Step 2: Apply hybrid strategy (resample + weights) on downsampled data
        if resample_config is None:
            resample_config = {'target_min_samples': 1000, 'noise_scale': 0.005}

        print(f"\nStep 2: Applying Hybrid Strategy (Resample + Weights) on downsampled data...")
        print(f"  - Oversampling minority classes to {resample_config['target_min_samples']} samples...")

        # Oversample minority classes
        X_resampled, y_resampled, _ = hybrid_balance_dataset(
            X_downsampled, y_downsampled,
            target_min_samples=resample_config['target_min_samples'],
            noise_scale=resample_config['noise_scale']
        )

        # Step 3: Compute additional class weights for fine-tuning
        print(f"\nStep 3: Computing class weights for remaining imbalance...")
        class_weights = compute_class_weights(
            y_resampled,
            strategy=weight_config.get('weight_strategy', 'balanced')
        )

        # Print final distribution
        y_resampled_np = y_resampled.cpu().numpy()
        counts_after = Counter(y_resampled_np)
        print("\nFinal Class Distribution (After Downsample + Hybrid):")
        for class_idx, count in sorted(counts_after.items()):
            try:
                class_name = label_encoder.classes_[class_idx] if class_idx < len(label_encoder.classes_) else f"Class_{class_idx}"
            except (IndexError, TypeError):
                class_name = f"Class_{class_idx}"
            print(f"  {class_name}: {count:6d} samples ({100*count/len(y_resampled_np):.2f}%)")
        print(f"\nFinal class weights: {class_weights.cpu().numpy()}")

        return X_resampled, y_resampled, class_weights

    elif strategy == 'none':
        # No balancing
        print("\nStrategy: No balancing (uniform weights)")
        return X_tensor, y_tensor, None

    else:
        raise ValueError(f"Unknown strategy: {strategy}. Choose from ['weights', 'resample', 'hybrid', 'smart', 'complete', 'downsample_weights', 'downsample_hybrid', 'none']")


In [ ]:
def main(
    balance_strategy: str = 'weights',
    resample_config: Optional[dict] = None,
    weight_config: Optional[dict] = None,
    use_optimized: bool = True,
    batch_size_videos: int = 50,
    window: int = 105
):
    """
    Main execution function. Loads data, processes videos, and trains model.

    ⚠️  WARNING: For large datasets (100+ videos), use main_optimized() or set use_optimized=True
        to prevent RAM overflow. The optimized version uses disk-based batch processing.

    Parameters
    ----------
    balance_strategy : str
        Strategy for handling imbalanced classes:
        - 'weights': Use class weights in loss (no resampling)
        - 'resample': Resample data to balance classes
        - 'hybrid': Combine resampling + weights
        - 'none': No balancing
    resample_config : dict, optional
        Configuration for resampling (see handle_imbalanced_data)
    weight_config : dict, optional
        Configuration for weight computation (see handle_imbalanced_data)
    use_optimized : bool
        If True (default), uses memory-optimized batch processing for large datasets.
        Recommended for 100+ videos to prevent RAM overflow.
    batch_size_videos : int
        Number of videos per batch when using optimized mode (default: 50)

    Returns
    -------
    tuple
        (X_tensor, y_tensor, label_encoder, scaler, master_training_set, class_weights)
    """
    # Load video metadata to check size
    try:
        videos_data = pd.read_csv('/content/drive/MyDrive/MABe-mouse-behavior-detection/train.csv')
        videos_data = videos_data.fillna(value='unknown')

        # Exclude specific labs that cause issues
        videos_data = videos_data[videos_data['lab_id'] != 'MABe22_keypoints']
        videos_data = videos_data[videos_data['lab_id'] != 'MABe22_movies']
        print(f"  Excluded labs: MABe22_keypoints, MABe22_movies")
    except FileNotFoundError as e:
        print(f"[Error] train.csv file not found: {e}")
        print("Returning empty results.")
        return None, None, None, None, pd.DataFrame(), None
    except Exception as e:
        print(f"[Error] Error reading train.csv: {e}")
        print("Returning empty results.")
        return None, None, None, None, pd.DataFrame(), None

    # Check number of videos and recommend optimized version
    num_videos = len(videos_data['video_id'].unique()) if 'video_id' in videos_data.columns else len(videos_data)

    if num_videos > 100 or use_optimized:
        if num_videos > 100:
            print(f"⚠️  Detected {num_videos} videos - automatically using optimized batch processing")
            print(f"   to prevent RAM overflow. Processing in batches of {batch_size_videos} videos.\n")

        return main_optimized(
            balance_strategy=balance_strategy,
            resample_config=resample_config,
            weight_config=weight_config,
            batch_size_videos=batch_size_videos,
            window=window,
            verbose=True
        )

    print(f"Processing {num_videos} videos (use main_optimized() for larger datasets)\n")

    # Original processing for small datasets
    data_config = iterate_video_metadata(videos_data)
    all_prossessed_datasets = []
    processed_count = 0

    for video_config in data_config:
        try:
            data_model = DataModeling(video_config)
            final_dataset = data_model.process(window=105, batch_size=15, skip_n=2)
            # Only append if dataset is not empty
            if not final_dataset.empty:
                all_prossessed_datasets.append({
                    'data': final_dataset,
                    'lab_id': video_config['lab_id'],
                    'video_id': video_config['video_id']
                })
            else:
                print(f"[Info] Skipping video {video_config['video_id']} (lab: {video_config['lab_id']}) - empty dataset.")

            processed_count += 1

            # Periodic memory cleanup
            if processed_count % 20 == 0:
                gc.collect()
                print(f"  Processed {processed_count} videos, Memory: {get_memory_usage_mb():.1f} MB")
        except FileNotFoundError as e:
            print(f"[Warning] File not found for video {video_config.get('video_id', 'unknown')} (lab: {video_config.get('lab_id', 'unknown')}): {e}")
            print("Continuing to next video...")
            continue
        except Exception as e:
            print(f"[Warning] Error processing video {video_config.get('video_id', 'unknown')} (lab: {video_config.get('lab_id', 'unknown')}): {e}")
            print("Continuing to next video...")
            continue

    # Aggregate datasets
    print(f"\nAggregating {len(all_prossessed_datasets)} processed datasets...")
    master_training_set = aggregate_multi_video_datasets(all_prossessed_datasets)

    # Clear intermediate data to free memory
    del all_prossessed_datasets
    gc.collect()
    print(f"Memory after aggregation: {get_memory_usage_mb():.1f} MB")

    # Prepare data for GRU
    X_tensor, y_tensor, label_encoder, scaler = prepare_data_for_gru(
        master_training_set, FEATURES, target_window_size=window
    )

    # Handle imbalanced data
    X_balanced, y_balanced, class_weights = handle_imbalanced_data(
        X_tensor, y_tensor,
        strategy=balance_strategy,
        resample_config=resample_config,
        weight_config=weight_config
    )

    print(f"Final memory usage: {get_memory_usage_mb():.1f} MB")

    return X_balanced, y_balanced, label_encoder, scaler, master_training_set, class_weights


def train_with_strategy(
    balance_strategy: str = 'weights',
    model_config: Optional[dict] = None,
    resample_config: Optional[dict] = None,
    weight_config: Optional[dict] = None,
    downsample_config: Optional[dict] = None,
    epochs: int = 50,
    learning_rate: float = 0.001,
    test_size: float = 0.2,
    random_state: int = 42,
    early_stopping_patience: Optional[int] = 10,
    evaluate: bool = True,
    apply_attack_rules: bool = False,
    attack_rules_config: Optional[dict] = None
):
    """
    Complete pipeline: Load data, handle imbalance, train model with stratified split, and evaluate.

    Parameters
    ----------
    balance_strategy : str
        Strategy for handling imbalanced classes:
        - 'weights': Use class weights in loss (fastest, no data modification) - RECOMMENDED
        - 'resample': Resample data to balance classes (slower, modifies data)
        - 'hybrid': Combine resampling + weights
        - 'smart': Intelligently downsample majority classes + oversample minority classes
        - 'complete': Complete strategy (Downsample + Resample + Weighting) - Most thorough
        - 'downsample_hybrid': Downsample majority classes + Hybrid (Resample + Weights)
        - 'none': No balancing (baseline)
    model_config : dict, optional
        Model configuration. If None, uses default config_model_1
    resample_config : dict, optional
        Resampling configuration:
        - target_min_samples: int (default: 1000, recommended: 500-1000 for production)
        - noise_scale: float (default: 0.005)
    weight_config : dict, optional
        Weight computation configuration:
        - weight_strategy: str (default: 'balanced')
          Options: 'balanced', 'inverse', 'sqrt'
    downsample_config : dict, optional
        Configuration for downsampling (required for 'smart' strategy):
        - 'classes_to_downsample': list of class names (e.g., ['other', 'co_other'])
        - 'downsample_percentage': float (0.0-1.0), percentage to keep (e.g., 0.3 = keep 30%)
        - 'method': str, 'random' or 'stratified' (default: 'random')
        - 'random_state': int, random seed (default: 42)
    epochs : int
        Number of training epochs
    learning_rate : float
        Learning rate for optimizer
    test_size : float
        Proportion of data for testing (default: 0.2)
    random_state : int
        Random seed for reproducibility
    early_stopping_patience : int, optional
        Number of epochs to wait before early stopping (default: 10)
    evaluate : bool
        Whether to evaluate model and print per-class metrics (default: True)
    apply_attack_rules : bool
        Whether to apply attack detection rules during evaluation (default: False)
    attack_rules_config : dict, optional
        Configuration for attack detection rules:
        - 'low_threshold': float (default: 0.3) - Low probability threshold
        - 'context_window': int (default: 3) - Number of surrounding frames
        - 'attack_class_names': list (default: None) - Auto-detects if None

    Returns
    -------
    tuple
        (model, label_encoder, scaler, class_weights, eval_results)
        eval_results is None if evaluate=False
    """
    # Default resample config with lower target_min_samples (recommended)
    if resample_config is None:
        resample_config = {'target_min_samples': 1000, 'noise_scale': 0.005}

    # Load and prepare data
    print("="*60)
    print("STEP 1: Loading and Preparing Data")
    print("="*60)
    X_tensor, y_tensor, label_encoder, scaler, master_training_set, _ = main(
        balance_strategy='none',  # Get original data first
        resample_config=resample_config,
        weight_config=weight_config
    )

In [ ]:
# =========================================================================
    # AUTO-CONFIGURE MODEL BASED ON DATA
    # =========================================================================
    # Infer input_size from data shape: X_tensor is (samples, seq_len, features)
    inferred_input_size = X_tensor.shape[2] if len(X_tensor.shape) == 3 else len(FEATURES)
    inferred_seq_len = X_tensor.shape[1] if len(X_tensor.shape) == 3 else 105

    # Infer output_size from number of classes
    inferred_output_size = len(label_encoder.classes_)

    # Build default config if not provided
    if model_config is None:
        model_config = {
            'hidden_size': inferred_seq_len,  # Use sequence length as hidden size
            'batch_size': 15,
            'input_size': inferred_input_size,
            'output_size': inferred_output_size,
            'bias': True,
            'bidirectional': False
        }
        print(f"\n  [Auto-Config] Model configuration inferred from data:")
        print(f"    - input_size:  {inferred_input_size} (from {len(FEATURES)} features)")
        print(f"    - output_size: {inferred_output_size} (from {list(label_encoder.classes_)})")
        print(f"    - hidden_size: {inferred_seq_len} (from sequence length)")
        print(f"    - batch_size:  15")
    else:
        # User provided config - validate and auto-fill missing values
        if 'input_size' not in model_config or model_config['input_size'] is None:
            model_config['input_size'] = inferred_input_size
            print(f"  [Auto-Config] input_size set to {inferred_input_size} (from data)")
        elif model_config['input_size'] != inferred_input_size:
            print(f"  [Warning] Config input_size ({model_config['input_size']}) != data features ({inferred_input_size})")
            print(f"  [Auto-Fix] Updating input_size to {inferred_input_size}")
            model_config['input_size'] = inferred_input_size

        if 'output_size' not in model_config or model_config['output_size'] is None:
            model_config['output_size'] = inferred_output_size
            print(f"  [Auto-Config] output_size set to {inferred_output_size} (from classes)")
        elif model_config['output_size'] != inferred_output_size:
            print(f"  [Warning] Config output_size ({model_config['output_size']}) != data classes ({inferred_output_size})")
            print(f"  [Auto-Fix] Updating output_size to {inferred_output_size}")
            model_config['output_size'] = inferred_output_size

        # Set defaults for optional params
        model_config.setdefault('hidden_size', inferred_seq_len)
        model_config.setdefault('batch_size', 15)
        model_config.setdefault('bias', True)
        model_config.setdefault('bidirectional', False)

    print(f"\n  Final model config: {model_config}")

    # Split data BEFORE balancing (stratified)
    print("\n" + "="*60)
    print("STEP 2: Stratified Train/Test Split")
    print("="*60)
    X_train_orig, X_test_orig, y_train_orig, y_test_orig = split_data_stratified(
        X_tensor, y_tensor, test_size=test_size, random_state=random_state
    )
    print(f"  Train set: {len(X_train_orig):,} samples")
    print(f"  Test set: {len(X_test_orig):,} samples")

    # Apply balancing strategy ONLY to training data
    print("\n" + "="*60)
    print("STEP 3: Handling Imbalanced Training Data")
    print("="*60)
    X_train_balanced, y_train_balanced, class_weights = handle_imbalanced_data(
        X_train_orig, y_train_orig,
        strategy=balance_strategy,
        resample_config=resample_config,
        weight_config=weight_config,
        label_encoder=label_encoder,  # Pass label encoder for smart strategy
        downsample_config=downsample_config  # Pass downsample config for smart strategy
    )

    # Train model with validation set
    print("\n" + "="*60)
    print("STEP 4: Training Model")
    print("="*60)
    model = train_gru_model(
        X_train_balanced, y_train_balanced, model_config,
        epochs=epochs,
        learning_rate=learning_rate,
        class_weights=class_weights,
        X_val=X_test_orig,  # Validate on original test set
        y_val=y_test_orig,
        early_stopping_patience=early_stopping_patience
    )

    # Evaluate on original test set (important for resampled strategies)
    eval_results = None
    if evaluate:
        print("\n" + "="*60)
        print("STEP 5: Evaluating on Original Test Set")
        print("="*60)
        print("(Using original imbalanced test set for realistic evaluation)")
        eval_results = evaluate_model_per_class(
            model, X_test_orig, y_test_orig, label_encoder
        )
        print_evaluation_results(eval_results, balance_strategy)

        # Optional: Apply attack detection rules for improved attack detection
        if apply_attack_rules:
            if attack_rules_config is None:
                attack_rules_config = {
                    'low_threshold': 0.3,
                    'context_window': 3,
                    'attack_class_names': None  # Auto-detect
                }

            print("\n" + "="*60)
            print("STEP 6: Applying Attack Detection Rules")
            print("="*60)
            eval_with_rules = evaluate_with_attack_rules(
                model, X_test_orig, y_test_orig, label_encoder,
                low_threshold=attack_rules_config.get('low_threshold', 0.3),
                context_window=attack_rules_config.get('context_window', 3),
                attack_class_names=attack_rules_config.get('attack_class_names', None)
            )
            # Update eval_results to include rules-based evaluation
            eval_results['with_attack_rules'] = eval_with_rules

        # NOTE: For catching rare attacks (attack/chase classes with 0% recall),
        # use BehavioralInferenceEngine after training. This addresses the "data ceiling"
        # problem where models cannot learn to detect rare attacks (8 training examples)
        # by accepting low-confidence predictions (10% instead of 90%) and validating
        # with behavioral context.
        #
        # Example usage:
        # eval_with_inference = evaluate_with_behavioral_inference(
        #     model, X_test_orig, y_test_orig, label_encoder,
        #     low_threshold=0.10,  # Low threshold to catch borderline attacks
        #     history_size=10      # Context memory (last 10 frames)
        # )

    return model, label_encoder, scaler, class_weights, eval_results


if __name__ == "__main__":
    # Example usage with different strategies:

    # Strategy 1: Use class weights only (RECOMMENDED for production)
    # - Fastest training, no data modification
    # - More realistic performance metrics
    # - Better generalization
    # Strategy 1: Use class weights only (RECOMMENDED for production)
    # - Fastest training, no data modification
    # - More realistic performance metrics
    # - Better generalization
    # - WITH ATTACK DETECTION RULES: Improves attack recall
    model1, label_encoder1, scaler1, weights1, eval1 = train_with_strategy(
        balance_strategy='weights',
        weight_config={'weight_strategy': 'balanced'},
        epochs=50,
        resample_config={'target_min_samples': 1000},  # Lower for production
        evaluate=True,
        apply_attack_rules=True,  # Enable attack detection rules
        attack_rules_config={
            'low_threshold': 0.3,  # Low threshold to catch borderline attacks
            'context_window': 3,   # Check 3 frames before/after
            'attack_class_names': ['attack', 'chaseattack']  # Specify attack classes
        }
    )

    # Strategy 2: Resample data (use with caution)
    # - Validates on original imbalanced test set for realistic evaluation
    # - Use lower target_min_samples (500-1000) for production
    # model2, label_encoder2, scaler2, weights2, eval2 = train_with_strategy(
    #     balance_strategy='resample',
    #     resample_config={'target_min_samples': 1000, 'noise_scale': 0.005},  # Lower for production
    #     epochs=50,
    #     evaluate=True  # Get per-class metrics
    # )

    # Strategy 3: Hybrid approach (resample + weights)
    # - Most thorough but computationally expensive
    # - Validates on original test set
    # model3, label_encoder3, scaler3, weights3, eval3 = train_with_strategy(
    #     balance_strategy='hybrid',
    #     resample_config={'target_min_samples': 1000, 'noise_scale': 0.005},  # Lower for production
    #     weight_config={'weight_strategy': 'balanced'},
    #     epochs=50,
    #     evaluate=True
    # )

    # Strategy 4: Smart balancing (Good for imbalanced data)
    # - Intelligently downsamples majority classes (e.g., 'other', 'co_other')
    # - Optionally oversamples minority classes
    # - More efficient than full resampling
    # model4, label_encoder4, scaler4, weights4, eval4 = train_with_strategy(
    #     balance_strategy='smart',
    #     downsample_config={
    #         'classes_to_downsample': ['other', 'co_other'],  # Majority classes to reduce
    #         'downsample_percentage': 0.3,  # Keep 30% of these classes (reduce by 70%)
    #         'method': 'random',
    #         'random_state': 42
    #     },
    #     resample_config={
    #         'target_min_samples': 1000,  # Oversample minority classes to this level
    #         'noise_scale': 0.005
    #     },
    #     epochs=50,
    #     evaluate=True
    # )

    # Strategy 5: Complete strategy (Downsample + Resample + Weighting)
    # - MOST THOROUGH: Combines all three techniques
    # - Step 1: Downsample majority classes (e.g., 'other', 'co_other')
    # - Step 2: Oversample minority classes to target level
    # - Step 3: Apply class weights for fine-tuning
    # - Best for maximum performance when you have compute resources
    # model5, label_encoder5, scaler5, weights5, eval5 = train_with_strategy(
    #     balance_strategy='complete',
    #     downsample_config={
    #         'classes_to_downsample': ['other', 'co_other'],  # Majority classes to reduce
    #         'downsample_percentage': 0.3,  # Keep 30% (reduce by 70%)
    #         'method': 'random',
    #         'random_state': 42
    #     },
    #     resample_config={
    #         'target_min_samples': 1000,  # Oversample minority classes to this level
    #         'noise_scale': 0.005
    #     },
    #     weight_config={
    #         'weight_strategy': 'balanced'  # Additional weighting for fine-tuning
    #     },
    #     epochs=50,
    #     evaluate=True
    # )

    # Strategy 6: Downsample + Weights (Efficient alternative)
    # - Step 1: Downsample majority classes (e.g., 'other', 'co_other')
    # - Step 2: Use class weights for remaining imbalance
    # - No oversampling (faster, smaller dataset)
    # - Good balance between efficiency and performance
    model6, label_encoder6, scaler6, weights6, eval6 = train_with_strategy(
         balance_strategy='downsample_weights',
         downsample_config={
             'classes_to_downsample': ['other', 'co_other'],  # Majority classes to reduce
             'downsample_percentage': 0.3,  # Keep 30% (reduce by 70%)
             'method': 'random',
             'random_state': 42
         },
         weight_config={
             'weight_strategy': 'balanced'  # Use weights for remaining imbalance
         },
         epochs=50,
         evaluate=True
     )

    # Strategy 7: Downsample + Hybrid (Downsample Majority + Resample + Weights)
    # - Step 1: Downsample majority classes (e.g., 'other', 'co_other')
    # - Step 2: Apply hybrid strategy (oversample minority + weights) on downsampled data
    # - Combines efficiency of downsampling with thoroughness of hybrid approach
    # - Good for imbalanced datasets where you want to reduce majority class first
    # model7, label_encoder7, scaler7, weights7, eval7 = train_with_strategy(
    #     balance_strategy='downsample_hybrid',
    #     downsample_config={
    #         'classes_to_downsample': ['other', 'co_other'],  # Majority classes to reduce
    #         'downsample_percentage': 0.3,  # Keep 30% (reduce by 70%)
    #         'method': 'random',
    #         'random_state': 42
    #     },
    #     resample_config={
    #         'target_min_samples': 1000,  # Oversample minority classes to this level
    #         'noise_scale': 0.005
    #     },
    #     weight_config={
    #         'weight_strategy': 'balanced'  # Additional weighting for fine-tuning
    #     },
    #     epochs=50,
    #     evaluate=True
    # )

    # Strategy 8: No balancing (baseline - not recommended)
    # - Likely to overfit to majority class
    # model5, label_encoder5, scaler5, weights5, eval5 = train_with_strategy(
    #     balance_strategy='none',
    #     epochs=50,
    #     evaluate=True
    # )


# --- CONFIGURATION: AUTO-CONFIGURED MODEL ---
# Set input_size and output_size to None for automatic configuration from data
# The train_with_strategy function will automatically infer these values:
#   - input_size: from X_tensor.shape[2] (number of features)
#   - output_size: from len(label_encoder.classes_) (number of classes)
#   - hidden_size: from sequence length (window size)

config_model_auto = {
    'hidden_size': None,      # Auto: will use sequence length (e.g., 105)
    'batch_size': 15,
    'input_size': None,       # Auto: will use number of features (e.g., 29)
    'output_size': None,      # Auto: will use number of classes
    'bias': True,
    'bidirectional': False
}

# Legacy config with hardcoded values (for reference)
config_model_1 = {
    'hidden_size': 105,
    'batch_size': 15,
    'input_size': 29,         # Must match len(FEATURES)
    'output_size': 13,        # Must match number of classes
    'bias': True,
    'bidirectional': False
}


def create_model_config(
    X_tensor: torch.Tensor = None,
    label_encoder: LabelEncoder = None,
    hidden_size: int = None,
    batch_size: int = 15,
    num_features: int = None,
    num_classes: int = None,
    bias: bool = True,
    bidirectional: bool = False
) -> dict:
    """
    Create a model configuration dictionary, automatically inferring values from data.

    Parameters
    ----------
    X_tensor : torch.Tensor, optional
        Input tensor of shape (samples, seq_len, features).
        If provided, input_size and hidden_size are inferred.
    label_encoder : LabelEncoder, optional
        Label encoder. If provided, output_size is inferred.
    hidden_size : int, optional
        Hidden layer size. If None, uses sequence length from X_tensor or 105.
    batch_size : int
        Batch size for training (default: 15)
    num_features : int, optional
        Number of input features. Overrides inference from X_tensor.
    num_classes : int, optional
        Number of output classes. Overrides inference from label_encoder.
    bias : bool
        Whether to use bias in GRU layers (default: True)
    bidirectional : bool
        Whether to use bidirectional GRU (default: False)

    Returns
    -------
    dict
        Model configuration dictionary with keys:
        - input_size, output_size, hidden_size, batch_size, bias, bidirectional

    Examples
    --------
    >>> # Auto-configure from data
    >>> config = create_model_config(X_tensor=X, label_encoder=le)

    >>> # Manual configuration
    >>> config = create_model_config(num_features=29, num_classes=13, hidden_size=128)

    >>> # Partial auto-config (override hidden_size)
    >>> config = create_model_config(X_tensor=X, label_encoder=le, hidden_size=256)
    """
    # Infer input_size
    if num_features is not None:
        input_size = num_features
    elif X_tensor is not None and len(X_tensor.shape) == 3:
        input_size = X_tensor.shape[2]
    else:
        input_size = len(FEATURES)  # Default to FEATURES list

    # Infer output_size
    if num_classes is not None:
        output_size = num_classes
    elif label_encoder is not None:
        output_size = len(label_encoder.classes_)
    else:
        output_size = None  # Will be set during training

    # Infer hidden_size
    if hidden_size is not None:
        final_hidden_size = hidden_size
    elif X_tensor is not None and len(X_tensor.shape) == 3:
        final_hidden_size = X_tensor.shape[1]  # Use sequence length
    else:
        final_hidden_size = 105  # Default window size

    config = {
        'input_size': input_size,
        'output_size': output_size,
        'hidden_size': final_hidden_size,
        'batch_size': batch_size,
        'bias': bias,
        'bidirectional': bidirectional
    }

    print(f"Model Config Created:")
    print(f"  - input_size:  {input_size} features")
    print(f"  - output_size: {output_size} classes")
    print(f"  - hidden_size: {final_hidden_size}")
    print(f"  - batch_size:  {batch_size}")
    print(f"  - bias:        {bias}")
    print(f"  - bidirectional: {bidirectional}")

    return config

# Example usage with imbalanced data handling:
#
# # Load and prepare data with balancing strategy
# X_tensor, y_tensor, label_encoder, scaler, master_training_set, class_weights = main(
#     balance_strategy='weights',  # Options: 'weights', 'resample', 'hybrid', 'none'
#     weight_config={'weight_strategy': 'balanced'}
# )
#
# # Train model with class weights
# model1 = train_gru_model(
#     X_tensor, y_tensor, config_model_1,
#     epochs=50,
#     learning_rate=0.001,
#     class_weights=class_weights  # Pass weights to use weighted loss
# )



# Example usage (uncomment to run):
# if __name__ == "__main__":
#     X_tensor, y_tensor, label_encoder, scaler, master_training_set = main()
#     model1 = train_gru_model(X_tensor, y_tensor, config_model_1, epochs=50)

# =============================================================================
# BEHAVIORAL INFERENCE ENGINE - USAGE EXAMPLE
# =============================================================================

def example_use_behavioral_inference_engine():
    """
    Example: How to use BehavioralInferenceEngine to catch attacks with low confidence.

    This is the "Magic Fix" that addresses the data ceiling problem where models
    cannot learn to detect rare attacks (8 training examples) by accepting
    low-confidence predictions (10% instead of 90%) and validating with context.

    Run this after training your best model (recommend Strategy 3 "Hybrid" or Strategy 4 "Complete").
    """
    print("\n" + "="*80)
    print("BEHAVIORAL INFERENCE ENGINE - USAGE EXAMPLE")
    print("="*80)
    print("\nThis example shows how to use the BehavioralInferenceEngine to catch")
    print("attacks that the model misses due to low training data (data ceiling problem).")
    print("\nThe engine uses:")
    print("  1. Low threshold (10%) to catch faint attack signals")
    print("  2. Context memory (last 10 frames) to validate predictions")
    print("  3. Risky context check (chase, approach, rear) to reduce false positives")
    print("="*80)

    # Example: Train a model first (use your best strategy)
    # model, label_encoder, scaler, weights, eval_results = train_with_strategy(
    #     balance_strategy='hybrid',  # or 'complete'
    #     epochs=50,
    #     evaluate=True
    # )

    # Then use BehavioralInferenceEngine for evaluation
    # eval_with_inference = evaluate_with_behavioral_inference(
    #     model, X_test, y_test, label_encoder,
    #     low_threshold=0.10,      # Low threshold (10%) to catch borderline attacks
    #     history_size=10,         # Remember last 10 predictions
    #     risky_context=['chase', 'co_rear', 'approach', 'co_chase', 'co_attack', 'rear']
    # )

    # Or use the engine directly for custom inference
    # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # engine = BehavioralInferenceEngine(
    #     model, device, label_encoder.classes_,
    #     low_threshold=0.10,
    #     history_size=10
    # )
    #
    # # Predict sequentially (maintains context)
    # predictions = engine.predict_sequential(X_test, batch_size=1, shuffle=False)

    print("\nSee the function definition above for full implementation details.")
    print("="*80 + "\n")


# Add analysis function at the end of the file
def analyze_training_results(results_dict: dict) -> None:
    """
    Analyze and compare training results from different strategies.

    Parameters
    ----------
    results_dict : dict
        Dictionary with strategy names as keys and tuples of (model, accuracy, loss) as values
    """
    print("\n" + "="*80)
    print("TRAINING RESULTS ANALYSIS")
    print("="*80)

    for strategy, (model, final_acc, final_loss, dataset_size) in results_dict.items():
        print(f"\n{strategy.upper()} Strategy:")
        print(f"  Final Accuracy: {final_acc:.2f}%")
        print(f"  Final Loss: {final_loss:.4f}")
        print(f"  Dataset Size: {dataset_size:,} samples")

    # Find best strategy
    best_strategy = max(results_dict.items(), key=lambda x: x[1][1])  # Sort by accuracy
    print(f"\n{'='*80}")
    print(f"BEST STRATEGY: {best_strategy[0].upper()}")
    print(f"  Accuracy: {best_strategy[1][1]:.2f}%")
    print(f"  Loss: {best_strategy[1][2]:.4f}")
    print(f"{'='*80}")

    # Recommendations
    print("\nRECOMMENDATIONS:")
    print("-" * 80)

    weights_acc = results_dict.get('weights', [None, 0, 0, 0])[1]
    resample_acc = results_dict.get('resample', [None, 0, 0, 0])[1]
    hybrid_acc = results_dict.get('hybrid', [None, 0, 0, 0])[1]
    none_acc = results_dict.get('none', [None, 0, 0, 0])[1]

    if none_acc > 99.5:
        print("⚠️  WARNING: 'none' strategy shows suspiciously high accuracy (>99.5%)")
        print("   This likely indicates overfitting to the majority class.")
        print("   The model may be predicting the dominant class for all samples.")

    if resample_acc > weights_acc + 5:
        print("✓ Resampling significantly improves performance over weights-only.")
        print("  Consider using 'resample' or 'hybrid' for production.")
    elif weights_acc > resample_acc - 2:
        print("✓ Weights-only performs nearly as well as resampling.")
        print("  Consider using 'weights' for faster training and smaller memory footprint.")

    if hybrid_acc > resample_acc:
        print("✓ Hybrid strategy (resample + weights) provides best results.")
    elif resample_acc > hybrid_acc:
        print("✓ Resampling alone is sufficient; additional weights don't help.")

    print("\nNOTE: High accuracy on imbalanced data may not reflect true performance.")
    print("      Consider using stratified train/test splits and per-class metrics.")
    print("      (Precision, Recall, F1-score per class)")